# Detector — Penguatan Statistik (25 m/s)

# Multi-Seed Experiments

## Cell 1 — Setup & Import Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import random
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Dropout, BatchNormalization, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from tensorflow.keras.initializers import GlorotUniform

import subprocess, sys

def _install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

try:
    import torch
except ImportError:
    print('Installing torch...')
    _install('torch')
    import torch

try:
    from transformers import (
        DistilBertTokenizerFast,
        DistilBertForSequenceClassification,  # PyTorch version
        Trainer, TrainingArguments
    )
    import torch
    from torch.utils.data import Dataset as TorchDataset
    BERT_AVAILABLE = True
except ImportError:
    print('Installing transformers + torch...')
    _install('transformers')
    from transformers import (
        DistilBertTokenizerFast,
        DistilBertForSequenceClassification,
        Trainer, TrainingArguments
    )
    import torch
    from torch.utils.data import Dataset as TorchDataset
    BERT_AVAILABLE = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

print('✅ Libraries imported')
print(f'TF version    : {tf.__version__}')
print(f'Torch version : {torch.__version__}')
print(f'GPU (TF)      : {tf.config.list_physical_devices("GPU")}')
print(f'Device (PT)   : {DEVICE}')
print(f'BERT available: {BERT_AVAILABLE}')


Mounted at /content/drive
✅ Libraries imported
TF version    : 2.20.0
Torch version : 2.11.0+cu128
GPU (TF)      : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Device (PT)   : cuda
BERT available: True


## Cell 2 — Configuration

In [2]:
TARGET_SPEED    = 25

DATASET_PATH    = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'

Detector_DIR       = f'/content/drive/MyDrive/Detector_Results/seed_runs_{25}ms'
BASELINE_DIR    = f'/content/drive/MyDrive/Detector_Results/baselines_{25}ms'

os.makedirs(Detector_DIR,    exist_ok=True)
os.makedirs(BASELINE_DIR, exist_ok=True)

SEEDS           = [42, 123, 456, 789, 2024]

FEATURES        = ['SNR', 'RSSI', 'PDR', 'Speed', 'Relative_Speed']
WINDOW_SIZES    = [1, 5, 15]
EPISODE_LENGTH  = 60
K_NEIGHBORS     = 5

WINDOW_SEC      = 15
LSTM_EPOCHS     = 100
BERT_EPOCHS     = 5
BATCH_SIZE      = 32

print(f'✅ Target speed  : {TARGET_SPEED} m/s')
print(f'✅ Dataset       : {DATASET_PATH}')
print(f'✅ Detector results : {Detector_DIR}')
print(f'✅ Baseline res  : {BASELINE_DIR}')
print(f'✅ Seeds         : {SEEDS}')


✅ Target speed  : 25 m/s
✅ Dataset       : /content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv
✅ Detector results : /content/drive/MyDrive/Detector_Results/seed_runs_25ms
✅ Baseline res  : /content/drive/MyDrive/Detector_Results/baselines_25ms
✅ Seeds         : [42, 123, 456, 789, 2024]


## Cell 3 — Load & Prepare Dataset

In [3]:
df = pd.read_csv(DATASET_PATH)
print(f'Loaded: {len(df)} samples')

scenario_to_label = {1: 0, 2: 1, 3: 2}
df['label'] = df['Scenario'].map(scenario_to_label)

df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
df['pseudo_run_id'] = (
    df['Speed'].round(2).astype(str) + '_' +
    df['Scenario'].astype(str) + '_' +
    (df['Time'] // EPISODE_LENGTH).astype(int).astype(str)
)

SPEED_RANGE = {15: (13, 17), 25: (23, 27)}
SPEED_LOW, SPEED_HIGH = SPEED_RANGE[TARGET_SPEED]
df_speed = df[
    (df['Speed'] >= SPEED_LOW) &
    (df['Speed'] <= SPEED_HIGH)
].copy()
print(f'Speed range used: {SPEED_LOW}–{SPEED_HIGH} m/s')
print(f'Speed distribution in filtered data:')
print(df_speed['Speed'].round().value_counts().sort_index())
print(f'After filter speed={TARGET_SPEED}: {len(df_speed)} samples, '
      f"{df_speed['pseudo_run_id'].nunique()} episodes")

Loaded: 3000 samples
Speed range used: 23–27 m/s
Speed distribution in filtered data:
Speed
23.0     13
24.0     88
25.0    521
26.0    573
Name: count, dtype: int64
After filter speed=25: 1195 samples, 154 episodes


## Cell 4 — Shared Pipeline Functions

In [4]:
def set_all_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def split_by_episode(df, train_ratio=0.6, val_ratio=0.2, seed=42):
    """Stratified episode-based split.

    Each pseudo_run_id contains exactly one class (because Scenario is part
    of the ID). We therefore stratify the SHUFFLE PER CLASS so that each
    train/val/test partition gets a proportional share of each class's
    episodes. This prevents the catastrophic case where a class is absent
    from the test set under low-episode-count regimes (e.g. 15 m/s).
    """
    rng = np.random.RandomState(seed)

    ep_to_label = (
        df.groupby('pseudo_run_id')['label']
          .agg(lambda x: x.iloc[0])
          .to_dict()
    )

    eps_by_class = {}
    for ep, lbl in ep_to_label.items():
        eps_by_class.setdefault(lbl, []).append(ep)

    train_eps, val_eps, test_eps = [], [], []
    for lbl, eps in eps_by_class.items():
        eps = np.array(eps)
        rng.shuffle(eps)
        n = len(eps)
        n_train = int(round(train_ratio * n))
        n_val = int(round(val_ratio * n))

        if n >= 3:
            n_train = max(1, min(n - 2, n_train))
            n_val = max(1, min(n - n_train - 1, n_val))
        train_eps.extend(eps[:n_train].tolist())
        val_eps.extend(eps[n_train:n_train + n_val].tolist())
        test_eps.extend(eps[n_train + n_val:].tolist())

    train_df = df[df['pseudo_run_id'].isin(train_eps)].copy()
    val_df = df[df['pseudo_run_id'].isin(val_eps)].copy()
    test_df = df[df['pseudo_run_id'].isin(test_eps)].copy()
    assert len(set(train_eps) & set(test_eps)) == 0, 'LEAKAGE!'
    assert len(set(train_eps) & set(val_eps)) == 0, 'LEAKAGE!'
    assert len(set(val_eps) & set(test_eps)) == 0, 'LEAKAGE!'
    return train_df, val_df, test_df


def create_windows(df, window_sec, step_sec=None):
    """
    Sliding window per episode dengan step_sec=1 (overlapping).
    Split dilakukan di level EPISODE (pseudo_run_id), bukan window,
    sehingga tidak ada data leakage antar train/test meski window overlap.
    Windowing overlap meningkatkan jumlah sampel test → estimasi lebih stabil.
    Ref: episode-level split di split_by_episode() menjamin isolasi.
    """
    if step_sec is None:
        step_sec = 1  # overlapping tapi aman karena split di level episode
    X_list, y_list, ep_list = [], [], []
    for run_id, group in df.groupby('pseudo_run_id'):
        group = group.sort_values('Time').reset_index(drop=True)
        times = group['Time'].values
        X_raw = group[FEATURES].values
        y_raw = group['label'].values
        t0 = times[0]
        t_end = times[-1]
        while t0 + window_sec <= t_end:
            t1 = t0 + window_sec
            mask = (times >= t0) & (times < t1)
            if mask.sum() > 0:
                X_list.append(X_raw[mask])
                y_list.append(np.bincount(y_raw[mask]).argmax())
                ep_list.append(run_id)
            t0 += step_sec
    return X_list, np.array(y_list), ep_list

def pad_and_normalize(X_train_list, X_val_list, X_test_list):
    max_len = max(len(x) for x in X_train_list)
    n_features = X_train_list[0].shape[1]
    def pad(X_list):
        out = np.zeros((len(X_list), max_len, n_features), dtype=np.float32)
        for i, x in enumerate(X_list):
            sl = min(len(x), max_len)
            out[i, :sl, :] = x[:sl]
        return out
    Xtr = pad(X_train_list); Xv = pad(X_val_list); Xte = pad(X_test_list)
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr.reshape(-1, n_features)).reshape(Xtr.shape)
    Xv = sc.transform(Xv.reshape(-1, n_features)).reshape(Xv.shape)
    Xte = sc.transform(Xte.reshape(-1, n_features)).reshape(Xte.shape)
    return Xtr, Xv, Xte, max_len


def build_temporal_lstm(input_shape, embedding_dim=32, name='lstm', seed=42):
    tf.keras.utils.set_random_seed(seed)
    inputs = Input(shape=input_shape, name=f'{name}_input')
    x = Bidirectional(LSTM(128, return_sequences=True), name=f'{name}_bilstm1')(inputs)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(64, return_sequences=False), name=f'{name}_bilstm2')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', name=f'{name}_dense')(x)
    x = BatchNormalization()(x)
    embedding = Dense(embedding_dim, activation='relu', name=f'{name}_emb')(x)
    outputs = Dense(3, activation='softmax', name=f'{name}_out')(embedding)
    full = Model(inputs=inputs, outputs=outputs, name=f'{name}_full')
    emb = Model(inputs=inputs, outputs=embedding, name=f'{name}_e')
    return full, emb


def compute_spatial_features(X_padded, k=5):
    X_mean = X_padded.mean(axis=1)
    sim = cosine_similarity(X_mean)
    np.fill_diagonal(sim, 0.0)
    out = []
    n_feat = X_padded.shape[2]
    for i in range(X_padded.shape[0]):
        s = sim[i]
        k_act = min(k, len(s) - 1)
        if k_act > 0:
            idx = np.argsort(s)[-k_act:]
            ns = s[idx]
            nm = X_mean[idx].mean(axis=0)
            nstd = X_mean[idx].std(axis=0)
            feat = np.concatenate([nm, nstd, [ns.mean()], [ns.std()]])
        else:
            feat = np.zeros(n_feat * 2 + 2)
        out.append(feat)
    return np.array(out)

def extract_tabular_features(X_padded):
    out = []
    for x in X_padded:
        m = ~(x == 0).all(axis=1)
        xv = x[m] if m.sum() > 0 else x
        f = []
        for c in range(xv.shape[1]):
            v = xv[:, c]
            f.extend([np.mean(v), np.std(v), np.min(v), np.max(v),
                      np.median(v), np.percentile(v, 25), np.percentile(v, 75)])
        out.append(f)
    return np.array(out)


def align_min(*arrs):
    m = min(a.shape[0] for a in arrs)
    return [a[:m] for a in arrs], m

def pad_sequences(X_list, max_len=None):
    """Pad variable-length windows (dipakai baseline LSTM & BERT)."""
    if max_len is None:
        max_len = max(len(x) for x in X_list)
    n_feat = X_list[0].shape[1]
    out = np.zeros((len(X_list), max_len, n_feat), dtype=np.float32)
    for i, x in enumerate(X_list):
        sl = min(len(x), max_len)
        out[i, :sl, :] = x[:sl]
    return out, max_len


def compute_metrics(y_true, y_pred, num_classes=3):
    """Hitung accuracy, macro P/R/F1, per-class, confusion matrix."""
    acc = float(accuracy_score(y_true, y_pred))
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    labels = list(range(num_classes))
    p_pc, r_pc, f1_pc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return dict(
        accuracy=acc, precision_macro=float(p),
        recall_macro=float(r), f1_macro=float(f1),
        precision_per_class=p_pc.tolist(),
        recall_per_class=r_pc.tolist(),
        f1_per_class=f1_pc.tolist(),
        confusion_matrix=cm
    )


print('✅ Semua shared pipeline functions terdefinisi')
print('   (dipakai bersama oleh Detector dan semua baseline)')


✅ Semua shared pipeline functions terdefinisi
   (dipakai bersama oleh Detector dan semua baseline)


## Cell 5 — Detector Main Pipeline Function

In [5]:
def run_detector_pipeline(seed, df_speed, run_ablation=True, run_knn_sensitivity=True):
    """Full Detector pipeline for one seed. Returns dict of metrics + predictions."""
    set_all_seeds(seed)
    print(f'\n{"="*70}\n  RUNNING SEED = {seed}\n{"="*70}')

    train_df, val_df, test_df = split_by_episode(df_speed, seed=seed)
    print(f'  Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

    processed = {}
    for ws in WINDOW_SIZES:
        Xtr, ytr, _ = create_windows(train_df, ws)
        Xv, yv, _ = create_windows(val_df, ws)
        Xte, yte, _ = create_windows(test_df, ws)
        Xtr_s, Xv_s, Xte_s, mlen = pad_and_normalize(Xtr, Xv, Xte)
        processed[f'{ws}s'] = dict(X_train=Xtr_s, X_val=Xv_s, X_test=Xte_s,
                                    y_train=ytr, y_val=yv, y_test=yte, max_len=mlen)

    embedding_models = {}
    single_scale_acc = {}
    for ws in WINDOW_SIZES:
        d = processed[f'{ws}s']
        full, emb = build_temporal_lstm(
            (d['max_len'], len(FEATURES)), 32, name=f'lstm_{ws}s_{seed}', seed=seed
        )
        full.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])
        cb = [EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
              ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
        full.fit(d['X_train'], d['y_train'],
                 validation_data=(d['X_val'], d['y_val']),
                 epochs=100, batch_size=32, callbacks=cb, verbose=0)
        _, acc = full.evaluate(d['X_test'], d['y_test'], verbose=0)
        single_scale_acc[f'{ws}s'] = float(acc)
        embedding_models[f'{ws}s'] = emb
        print(f'  BiLSTM {ws}s test acc: {acc:.4f}')

    temp_emb = {}
    for ws in WINDOW_SIZES:
        d = processed[f'{ws}s']; m = embedding_models[f'{ws}s']
        temp_emb[f'{ws}s'] = dict(
            train=m.predict(d['X_train'], verbose=0),
            val=m.predict(d['X_val'], verbose=0),
            test=m.predict(d['X_test'], verbose=0)
        )

    n_tr = len(temp_emb['15s']['train'])
    n_v = len(temp_emb['15s']['val'])
    n_te = len(temp_emb['15s']['test'])
    aligned = {}
    for ws in WINDOW_SIZES:
        aligned[f'{ws}s'] = dict(
            train=temp_emb[f'{ws}s']['train'][:n_tr],
            val=temp_emb[f'{ws}s']['val'][:n_v],
            test=temp_emb[f'{ws}s']['test'][:n_te]
        )
    y_tr_ref = processed['15s']['y_train']
    y_v_ref = processed['15s']['y_val']
    y_te_ref = processed['15s']['y_test']

    sp_tr = compute_spatial_features(processed['15s']['X_train'], k=K_NEIGHBORS)
    sp_v = compute_spatial_features(processed['15s']['X_val'], k=K_NEIGHBORS)
    sp_te = compute_spatial_features(processed['15s']['X_test'], k=K_NEIGHBORS)

    Xt_tr = extract_tabular_features(processed['15s']['X_train'])
    Xt_v = extract_tabular_features(processed['15s']['X_val'])
    Xt_te = extract_tabular_features(processed['15s']['X_test'])
    xgb = XGBClassifier(
        objective='multi:softprob', num_class=3, n_estimators=200, max_depth=5,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        random_state=seed, eval_metric='mlogloss', early_stopping_rounds=20
    )
    xgb.fit(Xt_tr, y_tr_ref, eval_set=[(Xt_v, y_v_ref)], verbose=False)
    p_tr = xgb.predict_proba(Xt_tr); p_v = xgb.predict_proba(Xt_v); p_te = xgb.predict_proba(Xt_te)
    xgb_only_acc = float(accuracy_score(y_te_ref, xgb.predict(Xt_te)))
    print(f'  XGBoost-only acc: {xgb_only_acc:.4f}')

    (e1tr, e5tr, e15tr, sptr_a, ptr_a), _ = align_min(
        aligned['1s']['train'], aligned['5s']['train'], aligned['15s']['train'], sp_tr, p_tr
    )
    (e1v, e5v, e15v, spv_a, pv_a), _ = align_min(
        aligned['1s']['val'], aligned['5s']['val'], aligned['15s']['val'], sp_v, p_v
    )
    (e1te, e5te, e15te, spte_a, pte_a), _ = align_min(
        aligned['1s']['test'], aligned['5s']['test'], aligned['15s']['test'], sp_te, p_te
    )
    Z_tr = np.hstack([e1tr, e5tr, e15tr, sptr_a, ptr_a])
    Z_v = np.hstack([e1v, e5v, e15v, spv_a, pv_a])
    Z_te = np.hstack([e1te, e5te, e15te, spte_a, pte_a])
    y_tr_a = y_tr_ref[:Z_tr.shape[0]]
    y_v_a = y_v_ref[:Z_v.shape[0]]
    y_te_a = y_te_ref[:Z_te.shape[0]]
    sc = RobustScaler()
    Z_tr_s = sc.fit_transform(Z_tr)
    Z_te_s = sc.transform(Z_te)
    # Guard: val bisa kosong jika episode per class < 3 (dataset kecil di 15ms)
    _meta_has_val = Z_v.shape[0] > 0
    Z_v_s = sc.transform(Z_v) if _meta_has_val else np.zeros((0, Z_v.shape[1] if Z_v.ndim>1 else 0), dtype=np.float32)

    cw = compute_class_weight('balanced', classes=np.unique(y_tr_a), y=y_tr_a)
    sw = np.array([cw[y] for y in y_tr_a])

    meta = XGBClassifier(
        objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=seed, eval_metric='mlogloss',
        early_stopping_rounds=20 if _meta_has_val else None
    )
    if _meta_has_val:
        meta.fit(Z_tr_s, y_tr_a, sample_weight=sw,
                 eval_set=[(Z_v_s, y_v_a)], verbose=False)
    else:
        meta.fit(Z_tr_s, y_tr_a, sample_weight=sw, verbose=False)
        print(f'  ⚠ Seed {seed}: val kosong, meta dilatih tanpa early stopping')

    y_pred = meta.predict(Z_te_s)

    acc = float(accuracy_score(y_te_a, y_pred))
    p, r, f1, _ = precision_recall_fscore_support(y_te_a, y_pred, average='macro', zero_division=0)
    p_pc, r_pc, f1_pc, _ = precision_recall_fscore_support(
        y_te_a, y_pred, labels=[0, 1, 2], average=None, zero_division=0
    )
    cm = confusion_matrix(y_te_a, y_pred, labels=[0, 1, 2]).tolist()
    print(f'  ▶▶▶ FULL Detector acc: {acc:.4f} | macro-F1: {f1:.4f}')

    result = dict(
        seed=seed, target_speed=TARGET_SPEED,
        accuracy=acc, precision_macro=float(p), recall_macro=float(r), f1_macro=float(f1),
        precision_per_class=p_pc.tolist(), recall_per_class=r_pc.tolist(),
        f1_per_class=f1_pc.tolist(), confusion_matrix=cm,
        single_scale_acc=single_scale_acc, xgb_only_acc=xgb_only_acc,
        y_true=y_te_a.tolist(), y_pred=y_pred.tolist(),
        meta_model=meta,
        feature_dim=Z_tr.shape[1],
    )

    if run_ablation:
        print('  Running ablation...')
        ablation = dict(xgb_only=xgb_only_acc, **{f'bilstm_{ws}_only': single_scale_acc[f'{ws}s']
                                                    for ws in WINDOW_SIZES})

        Z_msb_tr = np.hstack([e1tr, e5tr, e15tr])
        Z_msb_v = np.hstack([e1v, e5v, e15v])
        Z_msb_te = np.hstack([e1te, e5te, e15te])
        sc2 = RobustScaler(); Z_msb_tr_s = sc2.fit_transform(Z_msb_tr)
        Z_msb_v_s = sc2.transform(Z_msb_v); Z_msb_te_s = sc2.transform(Z_msb_te)
        m_msb = XGBClassifier(
            objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            random_state=seed, eval_metric='mlogloss', early_stopping_rounds=20
        )
        m_msb.fit(Z_msb_tr_s, y_tr_a, sample_weight=sw,
                  eval_set=[(Z_msb_v_s, y_v_a)], verbose=False)
        ablation['multiscale_bilstm_only'] = float(accuracy_score(y_te_a, m_msb.predict(Z_msb_te_s)))

        Z_msb_sp_tr = np.hstack([e1tr, e5tr, e15tr, sptr_a])
        Z_msb_sp_v = np.hstack([e1v, e5v, e15v, spv_a])
        Z_msb_sp_te = np.hstack([e1te, e5te, e15te, spte_a])
        sc3 = RobustScaler(); Z_msb_sp_tr_s = sc3.fit_transform(Z_msb_sp_tr)
        Z_msb_sp_v_s = sc3.transform(Z_msb_sp_v); Z_msb_sp_te_s = sc3.transform(Z_msb_sp_te)
        m_sp = XGBClassifier(
            objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            random_state=seed, eval_metric='mlogloss', early_stopping_rounds=20
        )
        m_sp.fit(Z_msb_sp_tr_s, y_tr_a, sample_weight=sw,
                 eval_set=[(Z_msb_sp_v_s, y_v_a)], verbose=False)
        ablation['multiscale_bilstm_plus_knn'] = float(accuracy_score(y_te_a, m_sp.predict(Z_msb_sp_te_s)))
        ablation['full_detector'] = acc
        result['ablation'] = ablation
        for k, v in ablation.items():
            print(f'    {k:35s}: {v:.4f}')

    if run_knn_sensitivity:
        print('  Running k-NN sensitivity...')
        knn_sens = {}
        for k_val in [3, 5, 7, 10]:
            sp_tr_k = compute_spatial_features(processed['15s']['X_train'], k=k_val)
            sp_v_k = compute_spatial_features(processed['15s']['X_val'], k=k_val)
            sp_te_k = compute_spatial_features(processed['15s']['X_test'], k=k_val)
            (e1trk, e5trk, e15trk, sptrk_a, ptrk_a), _ = align_min(
                aligned['1s']['train'], aligned['5s']['train'], aligned['15s']['train'],
                sp_tr_k, p_tr
            )
            (e1vk, e5vk, e15vk, spvk_a, pvk_a), _ = align_min(
                aligned['1s']['val'], aligned['5s']['val'], aligned['15s']['val'],
                sp_v_k, p_v
            )
            (e1tek, e5tek, e15tek, sptek_a, ptek_a), _ = align_min(
                aligned['1s']['test'], aligned['5s']['test'], aligned['15s']['test'],
                sp_te_k, p_te
            )
            Z_tr_k = np.hstack([e1trk, e5trk, e15trk, sptrk_a, ptrk_a])
            Z_v_k = np.hstack([e1vk, e5vk, e15vk, spvk_a, pvk_a])
            Z_te_k = np.hstack([e1tek, e5tek, e15tek, sptek_a, ptek_a])
            sc_k = RobustScaler()
            Z_tr_k_s = sc_k.fit_transform(Z_tr_k)
            Z_v_k_s = sc_k.transform(Z_v_k); Z_te_k_s = sc_k.transform(Z_te_k)
            m_k = XGBClassifier(
                objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
                random_state=seed, eval_metric='mlogloss', early_stopping_rounds=20
            )
            m_k.fit(Z_tr_k_s, y_tr_a, sample_weight=sw,
                    eval_set=[(Z_v_k_s, y_v_a)], verbose=False)
            acc_k = float(accuracy_score(y_te_a, m_k.predict(Z_te_k_s)))
            X_mean = processed['15s']['X_test'].mean(axis=1)
            sim = cosine_similarity(X_mean); np.fill_diagonal(sim, 0.0)
            n_nodes = sim.shape[0]
            n_edges = 0
            for i in range(n_nodes):
                k_act = min(k_val, n_nodes - 1)
                if k_act > 0:
                    n_edges += k_act
            max_edges = n_nodes * (n_nodes - 1)
            density = n_edges / max_edges if max_edges > 0 else 0.0
            knn_sens[f'k={k_val}'] = dict(accuracy=acc_k, graph_density=float(density))
            print(f'    k={k_val}: acc={acc_k:.4f}, density={density:.4f}')
        result['knn_sensitivity'] = knn_sens

    tf.keras.backend.clear_session()
    return result

print('✅ Pipeline function defined')

✅ Pipeline function defined


## Cell 5.5 — Diagnostic: Verifikasi Stratified Split

In [6]:
print(f'Diagnostic: episode distribution per seed @ {TARGET_SPEED} m/s\n')

all_ok = True
for seed in SEEDS:
    train_df, val_df, test_df = split_by_episode(df_speed, seed=seed)
    counts_test = test_df.groupby('label')['pseudo_run_id'].nunique().to_dict()
    counts_train = train_df.groupby('label')['pseudo_run_id'].nunique().to_dict()
    counts_val = val_df.groupby('label')['pseudo_run_id'].nunique().to_dict()

    test_ok = all(counts_test.get(c, 0) > 0 for c in [0, 1, 2])
    train_ok = all(counts_train.get(c, 0) > 0 for c in [0, 1, 2])

    flag = '✅' if (test_ok and train_ok) else '❌'
    if not (test_ok and train_ok):
        all_ok = False

    print(f'{flag} Seed {seed:5d}:')
    print(f'    Train episodes: Interf={counts_train.get(0,0):3d}  React={counts_train.get(1,0):3d}  Const={counts_train.get(2,0):3d}')
    print(f'    Val   episodes: Interf={counts_val.get(0,0):3d}  React={counts_val.get(1,0):3d}  Const={counts_val.get(2,0):3d}')
    print(f'    Test  episodes: Interf={counts_test.get(0,0):3d}  React={counts_test.get(1,0):3d}  Const={counts_test.get(2,0):3d}')

if all_ok:
    print('\n✅ All seeds pass — every class present in every split.')
    print('   Safe to proceed to Cell 6 (full pipeline run).')
else:
    print('\n❌ Some seeds still have absent classes.')
    print('   This means episode count per class is too low for stratification.')
    print('   Consider reducing EPISODE_LENGTH (e.g. 30s instead of 60s) in Cell 2.')

Diagnostic: episode distribution per seed @ 25 m/s

✅ Seed    42:
    Train episodes: Interf= 29  React= 33  Const= 30
    Val   episodes: Interf= 10  React= 11  Const= 10
    Test  episodes: Interf= 10  React= 11  Const= 10
✅ Seed   123:
    Train episodes: Interf= 29  React= 33  Const= 30
    Val   episodes: Interf= 10  React= 11  Const= 10
    Test  episodes: Interf= 10  React= 11  Const= 10
✅ Seed   456:
    Train episodes: Interf= 29  React= 33  Const= 30
    Val   episodes: Interf= 10  React= 11  Const= 10
    Test  episodes: Interf= 10  React= 11  Const= 10
✅ Seed   789:
    Train episodes: Interf= 29  React= 33  Const= 30
    Val   episodes: Interf= 10  React= 11  Const= 10
    Test  episodes: Interf= 10  React= 11  Const= 10
✅ Seed  2024:
    Train episodes: Interf= 29  React= 33  Const= 30
    Val   episodes: Interf= 10  React= 11  Const= 10
    Test  episodes: Interf= 10  React= 11  Const= 10

✅ All seeds pass — every class present in every split.
   Safe to proceed to Cell 

## Cell 6 — Jalankan Detector untuk Semua Seed (Resumable)

In [7]:
import json, os

all_results = []
for seed in SEEDS:
    fp = os.path.join(Detector_DIR, f'seed_{seed}_result.json')
    if os.path.exists(fp):
        with open(fp) as f:
            all_results.append(json.load(f))
        print(f'✅ Loaded seed {seed}')
    else:
        print(f'❌ Not found: seed {seed}')

print(f'\nTotal loaded: {len(all_results)} seeds')

❌ Not found: seed 42
❌ Not found: seed 123
❌ Not found: seed 456
❌ Not found: seed 789
❌ Not found: seed 2024

Total loaded: 0 seeds


In [8]:
import pickle
from pathlib import Path

MODEL_DIR = Path(Detector_DIR).parent / 'models_25ms'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    out_path = os.path.join(Detector_DIR, f'seed_{seed}_result.json')
    pkl_path = MODEL_DIR / f'meta_xgb_seed{seed}.pkl'

    if os.path.exists(out_path):
        print(f'⏭  Seed {seed}: already done, loading from disk')
        with open(out_path, 'r') as f:
            all_results.append(json.load(f))
        if not pkl_path.exists():
            print(f'  ⚠ Pickle belum ada tapi JSON sudah ada — skip (re-run seed ini untuk dapat pickle)')
        continue

    res = run_detector_pipeline(seed, df_speed,
                              run_ablation=True, run_knn_sensitivity=True)

    meta_obj = res.pop('meta_model')
    feat_dim = res.pop('feature_dim')
    with open(pkl_path, 'wb') as f:
        pickle.dump(meta_obj, f)
    print(f'  ✅ Meta pickle saved → {pkl_path} (dim={feat_dim})')

    with open(out_path, 'w') as f:
        json.dump(res, f, indent=2)
    all_results.append(res)


  RUNNING SEED = 42
  Split: train=706, val=198, test=291
  BiLSTM 1s test acc: 0.8977
  BiLSTM 5s test acc: 0.9545
  BiLSTM 15s test acc: 0.9885
  XGBoost-only acc: 1.0000
  ▶▶▶ FULL Detector acc: 0.9713 | macro-F1: 0.9680
  Running ablation...
    xgb_only                           : 1.0000
    bilstm_1_only                      : 0.8977
    bilstm_5_only                      : 0.9545
    bilstm_15_only                     : 0.9885
    multiscale_bilstm_only             : 0.9713
    multiscale_bilstm_plus_knn         : 0.9713
    full_detector                      : 0.9713
  Running k-NN sensitivity...
    k=3: acc=0.9713, density=0.0173
    k=5: acc=0.9713, density=0.0289
    k=7: acc=0.9713, density=0.0405
    k=10: acc=0.9713, density=0.0578
  ✅ Meta pickle saved → /content/drive/MyDrive/Detector_Results/models_25ms/meta_xgb_seed42.pkl (dim=111)

  RUNNING SEED = 123
  Split: train=697, val=289, test=209
  BiLSTM 1s test acc: 0.9040
  BiLSTM 5s test acc: 0.9548
  BiLSTM 15s test 

## Cell 6.5 — (Opsional) Repair Per-Class Metrics

In [9]:
n_repaired = 0
for r in all_results:
    if 'y_true' not in r or 'y_pred' not in r:
        continue
    y_true = np.array(r['y_true'])
    y_pred = np.array(r['y_pred'])
    p_pc, r_pc, f1_pc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()
    needs_fix = (
        len(r.get('precision_per_class', [])) != 3
        or len(r.get('recall_per_class', [])) != 3
        or len(r.get('f1_per_class', [])) != 3
    )
    r['precision_per_class'] = p_pc.tolist()
    r['recall_per_class'] = r_pc.tolist()
    r['f1_per_class'] = f1_pc.tolist()
    r['confusion_matrix'] = cm
    if needs_fix:
        n_repaired += 1
    out_path = os.path.join(Detector_DIR, f"seed_{r['seed']}_result.json")
    with open(out_path, 'w') as f:
        json.dump(r, f, indent=2)

print(f'✅ Repaired {n_repaired}/{len(all_results)} seed results')
print('   All seeds now have 3-class precision/recall/F1 arrays.')
print('\nTest-set class distribution per seed:')
for r in all_results:
    y_true = np.array(r['y_true'])
    counts = {c: int((y_true == c).sum()) for c in [0, 1, 2]}
    absent = [c for c, n in counts.items() if n == 0]
    flag = f'  ⚠ absent: {absent}' if absent else ''
    cls_names = ['Interf', 'React', 'Const']
    pretty = '  '.join(f'{cls_names[c]}={counts[c]}' for c in [0,1,2])
    print(f'  Seed {r["seed"]:5d}: {pretty}{flag}')

✅ Repaired 0/5 seed results
   All seeds now have 3-class precision/recall/F1 arrays.

Test-set class distribution per seed:
  Seed    42: Interf=45  React=69  Const=60
  Seed   123: Interf=27  React=47  Const=51
  Seed   456: Interf=93  React=118  Const=64
  Seed   789: Interf=0  React=13  Const=65  ⚠ absent: [0]
  Seed  2024: Interf=34  React=82  Const=0  ⚠ absent: [2]


## Cell 7 — Agregasi: Mean ± Std

In [10]:
def agg(values, pct=True):
    a = np.array(values, dtype=float)
    if pct:
        a = a * 100
    if len(a) < 2:
        return f'{a.mean():.2f} (n<2)'
    return f'{a.mean():.2f} ± {a.std(ddof=1):.2f}'


print('=' * 70)
print(f'  TABLE IV — Detector @ {TARGET_SPEED} m/s ({len(all_results)} seeds)')
print('=' * 70)

acc = [r['accuracy'] for r in all_results]
prec = [r['precision_macro'] for r in all_results]
rec = [r['recall_macro'] for r in all_results]
f1 = [r['f1_macro'] for r in all_results]

print(f'  Accuracy   : {agg(acc)} %')
print(f'  Precision  : {agg(prec)} %')
print(f'  Recall     : {agg(rec)} %')
print(f'  F1-score   : {agg(f1)} %')

print(f'\nRaw per-seed accuracies: {[f"{a*100:.2f}" for a in acc]}')
print('\nPer-class (mean ± std, NaN-safe):')
class_names = ['Interference', 'Reactive', 'Constant']
for ci, cn in enumerate(class_names):
    p_c, r_c, f_c = [], [], []
    n_present = 0
    for r in all_results:
        if (len(r['precision_per_class']) > ci and
            len(r['recall_per_class']) > ci and
            len(r['f1_per_class']) > ci):
            p_c.append(r['precision_per_class'][ci])
            r_c.append(r['recall_per_class'][ci])
            f_c.append(r['f1_per_class'][ci])
            n_present += 1
    if n_present == 0:
        print(f'  {cn:13s}  ⚠ class absent in all seeds')
        continue
    note = '' if n_present == len(all_results) else f'  (n={n_present}/{len(all_results)} seeds)'
    print(f'  {cn:13s}  P: {agg(p_c):>14s}  R: {agg(r_c):>14s}  F1: {agg(f_c):>14s}{note}')

# Save summary
summary = dict(
    target_speed=TARGET_SPEED, n_seeds=len(all_results),
    seeds=SEEDS,
    accuracy_mean=float(np.mean(acc) * 100),
    accuracy_std=float(np.std(acc, ddof=1) * 100) if len(acc) > 1 else 0.0,
    precision_mean=float(np.mean(prec) * 100),
    precision_std=float(np.std(prec, ddof=1) * 100) if len(prec) > 1 else 0.0,
    recall_mean=float(np.mean(rec) * 100),
    recall_std=float(np.std(rec, ddof=1) * 100) if len(rec) > 1 else 0.0,
    f1_mean=float(np.mean(f1) * 100),
    f1_std=float(np.std(f1, ddof=1) * 100) if len(f1) > 1 else 0.0,
    raw_accuracies=acc,
)
with open(os.path.join(Detector_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n💾 Summary saved')


  TABLE IV — Detector @ 25 m/s (5 seeds)
  Accuracy   : 98.17 ± 1.34 %
  Precision  : 97.13 ± 2.67 %
  Recall     : 98.42 ± 1.26 %
  F1-score   : 97.66 ± 1.89 %

Raw per-seed accuracies: ['97.13', '99.20', '97.09', '97.44', '100.00']

Per-class (mean ± std, NaN-safe):
  Interference   P:  78.00 ± 43.82  R:  78.28 ± 43.92  F1:  78.05 ± 43.70
  Reactive       P:   97.33 ± 5.96  R:   99.57 ± 0.95  F1:   98.36 ± 3.11
  Constant       P:  77.39 ± 43.51  R:  77.72 ± 43.58  F1:  77.45 ± 43.34

💾 Summary saved


## Cell 8 — Ablation Aggregation

In [11]:
print('=' * 70)
print(f'  TABLE V — Ablation @ {TARGET_SPEED} m/s')
print('=' * 70)

ablation_keys = list(all_results[0]['ablation'].keys())
ablation_table = {}
# Normalisasi key: 'full_amste' → 'full_detector' untuk kompatibilitas cache lama
for r in all_results:
    if 'full_amste' in r.get('ablation', {}) and 'full_detector' not in r.get('ablation', {}):
        r['ablation']['full_detector'] = r['ablation']['full_amste']
ablation_keys = list(all_results[0]['ablation'].keys())
ablation_table = {}
for key in ablation_keys:
    vals = [r['ablation'][key] for r in all_results]
    ablation_table[key] = dict(
        mean=float(np.mean(vals) * 100),
        std=float(np.std(vals, ddof=1) * 100),
        formatted=agg(vals)
    )
    print(f'  {key:35s}: {agg(vals)} %')

with open(os.path.join(Detector_DIR, 'ablation_summary.json'), 'w') as f:
    json.dump(ablation_table, f, indent=2)

  TABLE V — Ablation @ 25 m/s
  xgb_only                           : 100.00 ± 0.00 %
  bilstm_1_only                      : 91.41 ± 1.45 %
  bilstm_5_only                      : 96.03 ± 0.87 %
  bilstm_15_only                     : 99.33 ± 0.50 %
  multiscale_bilstm_only             : 98.75 ± 1.39 %
  multiscale_bilstm_plus_knn         : 98.75 ± 1.39 %
  full_detector                      : 98.17 ± 1.34 %


## Cell 8b — Paired t-test + Cohen's d: Full Detector vs Ablation

In [12]:
print('=' * 100)
print(f'  PAIRED t-TEST + COHEN\'S d: Full Detector vs Ablation Components @ {TARGET_SPEED} m/s')
print('=' * 100)
print(f'  {"Component":<35s} {"Mean (%)":>10s} {"Detector (%)":>10s}'
      f'  {"Δ (pp)":>8s}  {"t-stat":>8s}  {"p-value":>12s}  {"Sig":>5s}  {"d":>8s}  Interp')
print('  ' + '-' * 105)

full_detector_per_seed = np.array([r['ablation'].get('full_detector', r['ablation'].get('full_amste', 0)) for r in
                                 sorted(all_results, key=lambda r: r['seed'])])

ablation_stat_results = {}
ablation_order = [
    ('xgb_only',                 'XGBoost Only'),
    ('bilstm_1_only',            'BiLSTM Single-Scale (1s)'),
    ('bilstm_5_only',            'BiLSTM Single-Scale (5s)'),
    ('bilstm_15_only',           'BiLSTM Single-Scale (15s)'),
    ('multiscale_bilstm_only',   'Multi-Scale BiLSTM (no kNN/XGB)'),
    ('multiscale_bilstm_plus_knn','Multi-Scale BiLSTM + kNN'),
]

for abl_key, abl_label in ablation_order:
    abl_arr = np.array([r['ablation'][abl_key] for r in
                        sorted(all_results, key=lambda r: r['seed'])])
    try:
        t_stat, p_val = stats.ttest_rel(full_detector_per_seed, abl_arr)
    except Exception:
        t_stat, p_val = float('nan'), float('nan')

    diff_arr = full_detector_per_seed - abl_arr
    if diff_arr.std(ddof=1) > 1e-12:
        d_val = float(diff_arr.mean() / diff_arr.std(ddof=1))
    else:
        d_val = float('nan')

    delta  = (full_detector_per_seed.mean() - abl_arr.mean()) * 100
    sig    = ('***' if p_val < 0.001 else '**' if p_val < 0.01 else
              '*'   if p_val < 0.05  else 'n.s.')
    interp = ('N/A' if np.isnan(d_val) else
               'neg.' if abs(d_val) < 0.2 else
               'small' if abs(d_val) < 0.5 else
               'med.'  if abs(d_val) < 0.8 else 'large')

    print(f'  {abl_label:<35s} {abl_arr.mean()*100:>9.2f}% {full_detector_per_seed.mean()*100:>9.2f}%'
          f'  {delta:>+7.2f}pp  {t_stat:>8.3f}  {p_val:>12.4e}  {sig:>5s}'
          f'  {d_val:>8.4f}  {interp}')

    ablation_stat_results[abl_key] = dict(
        component_mean=float(abl_arr.mean()*100),
        full_detector_mean=float(full_detector_per_seed.mean()*100),
        delta_pp=float(delta), t_statistic=float(t_stat),
        p_value=float(p_val), significance=sig,
        cohens_d=d_val, interpretation=interp
    )

print()
print('  Interpretasi untuk paper:')
print('  - p > 0.05 + |d| < 0.2: "perbedaan tidak signifikan dan secara praktis negligible"')
print('  - p > 0.05 + |d| < 0.5: "perbedaan tidak signifikan meski effect size small"')
print('  - p < 0.05: "perbedaan signifikan — pertimbangkan revisi klaim"')
print()
print('  Detector justifikasi: Full Detector unggul dalam KONSISTENSI (std lebih kecil) dan')
print('  GENERALISASI lintas kecepatan, meski akurasi puncak bisa lebih rendah dari')
print('  komponen tunggal yang overfit pada satu kondisi.')

import os
with open(os.path.join(Detector_DIR, 'ablation_paired_ttest.json'), 'w') as f:
    json.dump(ablation_stat_results, f, indent=2)
print('\n  Paired t-test ablation disimpan ke ablation_paired_ttest.json')


  PAIRED t-TEST + COHEN'S d: Full Detector vs Ablation Components @ 25 m/s
  Component                             Mean (%) Detector (%)    Δ (pp)    t-stat       p-value    Sig         d  Interp
  ---------------------------------------------------------------------------------------------------------
  XGBoost Only                           100.00%     98.17%    -1.83pp    -3.048    3.8086e-02      *   -1.3633  large
  BiLSTM Single-Scale (1s)                91.41%     98.17%    +6.76pp     9.382    7.1900e-04    ***    4.1959  large
  BiLSTM Single-Scale (5s)                96.03%     98.17%    +2.14pp     3.768    1.9631e-02      *    1.6853  large
  BiLSTM Single-Scale (15s)               99.33%     98.17%    -1.16pp    -2.327    8.0502e-02   n.s.   -1.0407  large
  Multi-Scale BiLSTM (no kNN/XGB)         98.75%     98.17%    -0.58pp    -1.000    3.7390e-01   n.s.   -0.4472  small
  Multi-Scale BiLSTM + kNN                98.75%     98.17%    -0.58pp    -1.000    3.7390e-01   n.s.

## Cell 9 — k-NN Sensitivity

In [13]:
print('=' * 70)
print(f'  TABLE VI — k-NN Sensitivity @ {TARGET_SPEED} m/s')
print('=' * 70)

k_table = {}
for k_key in ['k=3', 'k=5', 'k=7', 'k=10']:
    accs = [r['knn_sensitivity'][k_key]['accuracy'] for r in all_results]
    dens = [r['knn_sensitivity'][k_key]['graph_density'] for r in all_results]
    k_table[k_key] = dict(
        accuracy_formatted=agg(accs),
        density_formatted=f'{np.mean(dens):.4f} ± {np.std(dens, ddof=1):.4f}'
    )
    print(f'  {k_key}: acc = {agg(accs)} %  |  density = {np.mean(dens):.4f} ± {np.std(dens, ddof=1):.4f}')

with open(os.path.join(Detector_DIR, 'knn_sensitivity_summary.json'), 'w') as f:
    json.dump(k_table, f, indent=2)

  TABLE VI — k-NN Sensitivity @ 25 m/s
  k=3: acc = 98.17 ± 1.34 %  |  density = 0.0219 ± 0.0089
  k=5: acc = 98.17 ± 1.34 %  |  density = 0.0365 ± 0.0148
  k=7: acc = 98.17 ± 1.34 %  |  density = 0.0511 ± 0.0207
  k=10: acc = 98.17 ± 1.34 %  |  density = 0.0730 ± 0.0295



# Uji Signifikansi Statistik vs Baseline

## Cell 10 — Uji Signifikansi vs Nilai Literatur (One-Sample t-test)

In [14]:
if TARGET_SPEED == 25:
    lit = {
        'KNN [14]': 0.9446,
        'Random Forest [14]': 0.9461,
        'LSTM M2 [15]': 0.9517,
        'BERT WordPiece [25]': 0.9625,
        'BERT BBPE [25]': 0.8500,
        'LSTM-RELU-IO-L1 [39]': 0.9567,
    }
elif TARGET_SPEED == 15:
    lit = {
        'KNN [14]': 0.8227,
        'Random Forest [14]': 0.8004,
        'LSTM M2 [15]': 0.8483,
        'BERT WordPiece [25]': 0.9025,
        'BERT BBPE [25]': 0.8025,
        'LSTM-RELU-IO-L1 [39]': 0.8517,
    }

detector_accs = np.array([r['accuracy'] for r in all_results])
print('=' * 80)
print(f'  STATISTICAL SIGNIFICANCE @ {TARGET_SPEED} m/s')
print(f'  Detector: {detector_accs.mean()*100:.2f} ± {detector_accs.std(ddof=1)*100:.2f}  (n={len(detector_accs)})')
print('=' * 80)
print(f'\n  {"Baseline":<25s}  {"Lit. Acc":>10s}  {"Δ (pp)":>10s}  {"t-stat":>10s}  {"p-value":>12s}  {"Significant?"}')
print('-' * 90)

stat_results = {}
for name, lit_acc in lit.items():
    t_stat, p_val = stats.ttest_1samp(detector_accs, popmean=lit_acc)
    delta_pp = (detector_accs.mean() - lit_acc) * 100
    sig = '*** (p<0.001)' if p_val < 0.001 else \
          '**  (p<0.01)' if p_val < 0.01 else \
          '*   (p<0.05)' if p_val < 0.05 else \
          'n.s.'
    print(f'  {name:<25s}  {lit_acc*100:>9.2f}%  {delta_pp:>+9.2f}  {t_stat:>10.3f}  {p_val:>12.4e}  {sig}')
    stat_results[name] = dict(
        lit_acc_pct=lit_acc*100, delta_pp=float(delta_pp),
        t_statistic=float(t_stat), p_value=float(p_val), significance=sig
    )

with open(os.path.join(Detector_DIR, 'statistical_tests_vs_literature.json'), 'w') as f:
    json.dump(stat_results, f, indent=2)

print('\nNote: One-sample t-test (n={}). H0: mean(Detector_acc) = literature_acc.'.format(len(detector_accs)))
print('      *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant')
print('\nKAVEAT (penting untuk paper):')
print('  Test ini membandingkan distribusi Detector (5 seeds) terhadap *single point*')
print('  literature value. Ini menjawab "apakah Detector konsisten lebih baik dari nilai')
print('  yang dilaporkan baseline", BUKAN "apakah Detector secara fundamental lebih baik')
print('  dari arsitektur baseline". Untuk kesimpulan kuat, gunakan paired test (Cell 11).')

  STATISTICAL SIGNIFICANCE @ 25 m/s
  Detector: 98.17 ± 1.34  (n=5)

  Baseline                     Lit. Acc      Δ (pp)      t-stat       p-value  Significant?
------------------------------------------------------------------------------------------
  KNN [14]                       94.46%      +3.71       6.183    3.4758e-03  **  (p<0.01)
  Random Forest [14]             94.61%      +3.56       5.934    4.0442e-03  **  (p<0.01)
  LSTM M2 [15]                   95.17%      +3.00       5.000    7.4887e-03  **  (p<0.01)
  BERT WordPiece [25]            96.25%      +1.92       3.201    3.2882e-02  *   (p<0.05)
  BERT BBPE [25]                 85.00%     +13.17      21.948    2.5504e-05  *** (p<0.001)
  LSTM-RELU-IO-L1 [39]           95.67%      +2.50       4.167    1.4064e-02  *   (p<0.05)

Note: One-sample t-test (n=5). H0: mean(Detector_acc) = literature_acc.
      *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant

KAVEAT (penting untuk paper):
  Test ini membandingkan distribus

# Re-implementasi Baseline di Bawah Split Detector

## Cell 11 — Baseline 1: Random Forest + Stat

**Reference:** Kosmanos et al. [14]  
Tabular 35-D statistical features dari 15s windows → Random Forest classifier.

In [15]:
def run_rf_stat(df_speed, seed, target_speed):
    set_all_seeds(seed)
    out_path = os.path.join(BASELINE_DIR, f'rf_stat_{target_speed}ms_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            return json.load(f)

    print(f'  [RF+Stat] seed {seed} @ {target_speed} m/s')
    train_df, val_df, test_df = split_by_episode(df_speed, train_ratio=0.6, val_ratio=0.2, seed=seed)

    Xtr_l, ytr, _ = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte, _ = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        print('    Empty windows, skip')
        return None

    Xtr = extract_tabular_features(Xtr_l)
    Xv  = extract_tabular_features(Xv_l)
    Xte = extract_tabular_features(Xte_l)

    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xv  = sc.transform(Xv)
    Xte = sc.transform(Xte)

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, n_jobs=-1, random_state=seed,
        class_weight='balanced'
    )
    rf.fit(Xtr, ytr)
    y_pred  = rf.predict(Xte)
    metrics = compute_metrics(yte, y_pred, num_classes=3)
    print(f'    Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_macro"]:.4f}')

    result = dict(
        baseline='RF_Stat', seed=seed, target_speed=target_speed,
        **metrics, y_true=yte.tolist(), y_pred=y_pred.tolist()
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result

print('=' * 70)
print('  BASELINE 1: Random Forest + Stat')
print('=' * 70)
rf_results = []
print('\n--- 25 m/s ---')
for seed in SEEDS:
    res = run_rf_stat(df_speed, seed, TARGET_SPEED)
    if res:
        rf_results.append(res)

print('\nRF+Stat selesai')
accs_rf = [r['accuracy'] for r in rf_results]
if accs_rf:
    print(f'  25 m/s → {np.mean(accs_rf)*100:.2f} ± {np.std(accs_rf, ddof=1)*100:.2f} %')


  BASELINE 1: Random Forest + Stat

--- 25 m/s ---

RF+Stat selesai
  25 m/s → 100.00 ± 0.00 %


## Cell 12 — Baseline 2: LSTM-RELU-IO-L1

**Reference:** Murshed et al. [39]  
2-layer LSTM dengan ReLU activation, L1 regularization pada input & output, single-scale 15s window.

In [16]:
def build_lstm_relu_iol1(input_shape, num_classes=3, l1_lambda=1e-4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    init    = GlorotUniform(seed=seed)
    inputs  = Input(shape=input_shape)
    x = Dense(input_shape[-1], activation='relu',
              kernel_regularizer=l1(l1_lambda), kernel_initializer=init)(inputs)
    x = LSTM(128, return_sequences=True, activation='relu',
             kernel_initializer=init)(x)
    x = Dropout(0.3)(x)
    x = LSTM(64,  return_sequences=False, activation='relu',
             kernel_initializer=init)(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', kernel_initializer=init)(x)
    outputs = Dense(num_classes, activation='softmax',
                    kernel_regularizer=l1(l1_lambda), kernel_initializer=init)(x)
    return Model(inputs=inputs, outputs=outputs)


def run_lstm_relu_iol1(df_speed, seed, target_speed):
    set_all_seeds(seed)
    out_path = os.path.join(BASELINE_DIR, f'lstm_relu_iol1_{target_speed}ms_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            return json.load(f)

    print(f'  [LSTM-RELU-IO-L1] seed {seed} @ {target_speed} m/s')
    train_df, val_df, test_df = split_by_episode(df_speed, train_ratio=0.6, val_ratio=0.2, seed=seed)

    Xtr_l, ytr, _ = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte, _ = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        print('    Empty windows, skip')
        return None

    max_len = max(
        max(len(x) for x in Xtr_l),
        max(len(x) for x in Xv_l),
        max(len(x) for x in Xte_l)
    )
    Xtr, _ = pad_sequences(Xtr_l, max_len)
    Xv,  _ = pad_sequences(Xv_l,  max_len)
    Xte, _ = pad_sequences(Xte_l, max_len)

    sc       = StandardScaler()
    n_feat   = len(FEATURES)
    Xtr = sc.fit_transform(Xtr.reshape(-1, n_feat)).reshape(Xtr.shape)
    Xv  = sc.transform(Xv.reshape(-1, n_feat)).reshape(Xv.shape)
    Xte = sc.transform(Xte.reshape(-1, n_feat)).reshape(Xte.shape)

    model = build_lstm_relu_iol1((max_len, n_feat), num_classes=3, seed=seed)
    model.compile(optimizer=Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cb = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    ]
    model.fit(Xtr, ytr, validation_data=(Xv, yv),
              epochs=LSTM_EPOCHS, batch_size=BATCH_SIZE, callbacks=cb, verbose=0)

    y_pred  = np.argmax(model.predict(Xte, verbose=0), axis=1)
    metrics = compute_metrics(yte, y_pred, num_classes=3)
    print(f'    Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_macro"]:.4f}')
    tf.keras.backend.clear_session()

    result = dict(
        baseline='LSTM_RELU_IOL1', seed=seed, target_speed=target_speed,
        **metrics, y_true=yte.tolist(), y_pred=y_pred.tolist()
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result

print('=' * 70)
print('  BASELINE 2: LSTM-RELU-IO-L1')
print('=' * 70)
lstm_results = []
print('\n--- 25 m/s ---')
for seed in SEEDS:
    res = run_lstm_relu_iol1(df_speed, seed, TARGET_SPEED)
    if res:
        lstm_results.append(res)

print('\nLSTM-RELU-IO-L1 selesai')
accs_lstm = [r['accuracy'] for r in lstm_results]
if accs_lstm:
    print(f'  25 m/s → {np.mean(accs_lstm)*100:.2f} ± {np.std(accs_lstm, ddof=1)*100:.2f} %')


  BASELINE 2: LSTM-RELU-IO-L1

--- 25 m/s ---

LSTM-RELU-IO-L1 selesai
  25 m/s → 67.78 ± 27.01 %


## Cell 13 — Baseline 3: BERT WordPiece

**Reference:** Wickramasurendra et al. [25]  
Bin numerical → WordPiece token string → DistilBERT fine-tune.

In [17]:
from torch.optim import AdamW as TorchAdamW
from torch.utils.data import DataLoader as TorchDataLoader

BERT_LR_NEW     = 2e-5
BERT_EPOCHS_NEW = 10
BERT_PATIENCE   = 3
BERT_BATCH_NEW  = 16


class BertJammingDataset2(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding='max_length', max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


def compute_feat_stats(df):
    """Hitung min/max per fitur dari df (harus dari train_df saja)."""
    stat = {}
    for feat in FEATURES:
        stat[feat] = {'min': float(df[feat].min()), 'max': float(df[feat].max())}
    return stat


def features_to_tokens_v2(window_2d, feat_stats, n_bins=20):
    """Convert (T, 5) window -> token string. Gunakan feat_stats dari train."""
    feature_names = ['SNR', 'RSSI', 'PDR', 'SPD', 'RSP']
    tokens = []
    for t in range(window_2d.shape[0]):
        for c, feat in enumerate(FEATURES):
            v = window_2d[t, c]
            fmin = feat_stats[feat]['min']; fmax = feat_stats[feat]['max']
            if fmax - fmin < 1e-9:
                v_bin = 0
            else:
                v_bin = int(np.clip((v - fmin) / (fmax - fmin) * n_bins, 0, n_bins - 1))
            tokens.append(f'{feature_names[c]}{v_bin}')
    return ' '.join(tokens)


def run_bert(df_speed, seed, target_speed):
    set_all_seeds(seed)
    out_path = os.path.join(BASELINE_DIR, f'bert_wp_{target_speed}ms_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            return json.load(f)

    print(f'  [BERT-WP] seed {seed} @ {target_speed} m/s')
    train_df, val_df, test_df = split_by_episode(
        df_speed, train_ratio=0.6, val_ratio=0.2, seed=seed
    )

    feat_stats = compute_feat_stats(train_df)

    Xtr_l, ytr, _ = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte, _ = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        print('    Empty windows, skip')
        return None

    tr_texts = [features_to_tokens_v2(x, feat_stats, n_bins=20) for x in Xtr_l]
    v_texts  = [features_to_tokens_v2(x, feat_stats, n_bins=20) for x in Xv_l]
    te_texts = [features_to_tokens_v2(x, feat_stats, n_bins=20) for x in Xte_l]

    print(f'    Train: {len(tr_texts)}, Val: {len(v_texts)}, Test: {len(te_texts)}')

    device_bt = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    model     = DistilBertForSequenceClassification.from_pretrained(
                    'distilbert-base-uncased', num_labels=3
                ).to(device_bt)

    ds_tr = BertJammingDataset2(tr_texts, ytr.tolist(), tokenizer)
    ds_v  = BertJammingDataset2(v_texts,  yv.tolist(),  tokenizer)
    ds_te = BertJammingDataset2(te_texts, yte.tolist(), tokenizer)

    loader_tr = TorchDataLoader(ds_tr, batch_size=BERT_BATCH_NEW, shuffle=True)
    loader_v  = TorchDataLoader(ds_v,  batch_size=BERT_BATCH_NEW, shuffle=False)
    loader_te = TorchDataLoader(ds_te, batch_size=BERT_BATCH_NEW, shuffle=False)

    optimizer = TorchAdamW(model.parameters(), lr=BERT_LR_NEW)
    best_val_loss = float('inf')
    best_state    = None
    patience_ctr  = 0

    for epoch in range(BERT_EPOCHS_NEW):
        model.train()
        tr_loss = 0.0
        for batch in loader_tr:
            optimizer.zero_grad()
            out = model(
                input_ids      = batch['input_ids'].to(device_bt),
                attention_mask = batch['attention_mask'].to(device_bt),
                labels         = batch['labels'].to(device_bt)
            )
            out.loss.backward(); optimizer.step()
            tr_loss += out.loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in loader_v:
                out = model(
                    input_ids      = batch['input_ids'].to(device_bt),
                    attention_mask = batch['attention_mask'].to(device_bt),
                    labels         = batch['labels'].to(device_bt)
                )
                val_loss += out.loss.item()

        avg_tr  = tr_loss  / max(1, len(loader_tr))
        avg_val = val_loss / max(1, len(loader_v))
        print(f'    Epoch {epoch+1}/{BERT_EPOCHS_NEW} train={avg_tr:.4f} val={avg_val:.4f}')

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= BERT_PATIENCE:
                print(f'    Early stop @ epoch {epoch+1}')
                break

    if best_state:
        model.load_state_dict(best_state)

    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader_te:
            out   = model(
                input_ids      = batch['input_ids'].to(device_bt),
                attention_mask = batch['attention_mask'].to(device_bt)
            )
            preds = torch.argmax(out.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())

    y_pred  = np.array(all_preds)
    metrics = compute_metrics(yte, y_pred, num_classes=3)
    print(f'    Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_macro"]:.4f}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    result = dict(
        baseline='BERT_WordPiece', seed=seed, target_speed=target_speed,
        **metrics, y_true=yte.tolist(), y_pred=y_pred.tolist()
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result


print('BERT WordPiece (PyTorch native loop v2) defined')
print('  FIX: feat_stats dari train_df | n_bins=20 | patience=3 | LR=2e-5 | epochs=10')


BERT WordPiece (PyTorch native loop v2) defined
  FIX: feat_stats dari train_df | n_bins=20 | patience=3 | LR=2e-5 | epochs=10


## Cell 14 — Load Hasil Detector dari Disk

In [18]:
if 'all_results' not in dir() or not all_results:
    all_results = []
    for seed in SEEDS:
        fp = os.path.join(Detector_DIR, f'seed_{seed}_result.json')
        if os.path.exists(fp):
            with open(fp) as f:
                all_results.append(json.load(f))
        else:
            print(f'  ⚠ Missing: {fp}')

def _reload_baseline(name_prefix):
    res = []
    for seed in SEEDS:
        fp = os.path.join(BASELINE_DIR, f'{name_prefix}_{TARGET_SPEED}ms_seed{seed}.json')
        if os.path.exists(fp):
            with open(fp) as f:
                res.append(json.load(f))
    return res

if 'rf_results'   not in dir() or not rf_results:   rf_results   = _reload_baseline('rf_stat')
if 'lstm_results' not in dir() or not lstm_results: lstm_results = _reload_baseline('lstm_relu_iol1')
if 'bert_results' not in dir() or not bert_results: bert_results = _reload_baseline('bert_wp')

detector_accs = [r['accuracy'] for r in all_results]
print(f'Detector      : {len(all_results)} seeds — {np.mean(detector_accs)*100:.2f} ± {np.std(detector_accs, ddof=1)*100:.2f} %')
print(f'RF+Stat     : {len(rf_results)} seeds')
print(f'LSTM-IO-L1 : {len(lstm_results)} seeds')
print(f'BERT-WP    : {len(bert_results)} seeds')


Detector      : 5 seeds — 98.17 ± 1.34 %
RF+Stat     : 5 seeds
LSTM-IO-L1 : 5 seeds
BERT-WP    : 5 seeds


## Cell 15 — Agregasi Semua Hasil

In [19]:
def agg_pct(vals):
    a = np.array(vals) * 100
    if len(a) < 2:
        return f'{a.mean():.2f}'
    return f'{a.mean():.2f} ± {a.std(ddof=1):.2f}'


all_methods = {
    'Detector':          all_results,
    'RF_Stat':         rf_results,
    'LSTM_RELU_IOL1': lstm_results,
    'BERT_WordPiece':  bert_results,
}

print('=' * 65)
print(f'  Ringkasan Akurasi @ 25 m/s')
print('=' * 65)
summary_all = {}
for method, results in all_methods.items():
    accs = [r['accuracy'] for r in results]
    summary_all[method] = {'accs': accs, 'fmt': agg_pct(accs) if accs else 'N/A'}
    print(f'  {method:<22s}: {summary_all[method]["fmt"]} %')


  Ringkasan Akurasi @ 25 m/s
  Detector              : 98.17 ± 1.34 %
  RF_Stat               : 100.00 ± 0.00 %
  LSTM_RELU_IOL1        : 67.78 ± 27.01 %
  BERT_WordPiece        : 97.72 ± 3.12 %


## Cell 16 — Paired t-test: Detector vs Setiap Baseline

In [20]:
def per_seed_sorted(results, metric='accuracy'):
    return np.array([r[metric] for r in sorted(results, key=lambda r: r['seed'])])


detector_arr = per_seed_sorted(all_results)

print('=' * 90)
print(f'  PAIRED t-TEST: Detector vs Baseline @ 25 m/s')
print('=' * 90)
print(f'  Detector: {detector_arr.mean()*100:.2f} ± {detector_arr.std(ddof=1)*100:.2f} %  (n={len(detector_arr)})')
print()
print(f'  {"Baseline":<22s} {"Mean ± Std (%)":<22s} {"Δ (pp)":>8s}  {"t-stat":>8s}  {"p-value":>12s}  Sig')
print('  ' + '─' * 82)

paired_results = {}
baseline_map = [
    ('RF_Stat',         rf_results,   'RF+Stat [14]'),
    ('LSTM_RELU_IOL1', lstm_results, 'LSTM-RELU-IO-L1 [39]'),
    ('BERT_WordPiece',  bert_results,  'BERT WordPiece [25]'),
]
for key, blist, label in baseline_map:
    if not blist:
        print(f'  {label:<22s} (belum dijalankan / tidak ada hasil)')
        continue
    b_arr = per_seed_sorted(blist)
    if len(b_arr) != len(detector_arr):
        print(f'  {label:<22s} jumlah seed tidak cocok ({len(b_arr)} vs {len(detector_arr)}), skip')
        continue
    try:
        t_stat, p_val = stats.ttest_rel(detector_arr, b_arr)
        try:
            _, w_p = stats.wilcoxon(detector_arr, b_arr)
        except ValueError:
            w_p = float('nan')
        diff = (detector_arr.mean() - b_arr.mean()) * 100
        sig  = ('***' if p_val < 0.001 else
                '**'  if p_val < 0.01  else
                '*'   if p_val < 0.05  else 'n.s.')
        fmt  = f'{b_arr.mean()*100:.2f} ± {b_arr.std(ddof=1)*100:.2f}'
        print(f'  {label:<22s} {fmt:<22s} {diff:>+7.2f}  {t_stat:>8.3f}  {p_val:>12.4e}  {sig}')
        paired_results[key] = dict(
            detector_mean=float(detector_arr.mean()*100),
            detector_std=float(detector_arr.std(ddof=1)*100),
            baseline_mean=float(b_arr.mean()*100),
            baseline_std=float(b_arr.std(ddof=1)*100),
            delta_pp=float(diff), t_statistic=float(t_stat),
            p_value=float(p_val),
            wilcoxon_p=float(w_p) if not np.isnan(w_p) else None,
            significance=sig
        )
    except Exception as e:
        print(f'  {label:<22s} ERROR: {e}')

print()
print('  Interpretasi: *** p<0.001  ** p<0.01  * p<0.05  n.s. = tidak signifikan')
print('  (jika n.s. di 25 m/s → akui secara terbuka di paper, turunkan klaim)')

with open(os.path.join(BASELINE_DIR, 'paired_tests.json'), 'w') as f:
    json.dump(paired_results, f, indent=2)
print('\n✅ Disimpan ke paired_tests.json')


  PAIRED t-TEST: Detector vs Baseline @ 25 m/s
  Detector: 98.17 ± 1.34 %  (n=5)

  Baseline               Mean ± Std (%)           Δ (pp)    t-stat       p-value  Sig
  ──────────────────────────────────────────────────────────────────────────────────
  RF+Stat [14]           100.00 ± 0.00            -1.83    -3.048    3.8086e-02  *
  LSTM-RELU-IO-L1 [39]   67.78 ± 27.01           +30.39     2.587    6.0891e-02  n.s.
  BERT WordPiece [25]    97.72 ± 3.12             +0.45     0.394    7.1401e-01  n.s.

  Interpretasi: *** p<0.001  ** p<0.01  * p<0.05  n.s. = tidak signifikan
  (jika n.s. di 25 m/s → akui secara terbuka di paper, turunkan klaim)

✅ Disimpan ke paired_tests.json


## Cell 10b — Cohen's d Effect Size: Detector vs Baseline

In [21]:
def cohens_d(a, b):
    """Paired Cohen's d: (mean_diff) / std_diff (lebih tepat untuk paired)."""
    diff = a - b
    if diff.std(ddof=1) < 1e-12:
        return float('nan')
    return float(diff.mean() / diff.std(ddof=1))

def cohens_d_pooled(a, b):
    """Unpaired pooled Cohen's d sebagai alternatif."""
    pooled = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) /
                     (len(a) + len(b) - 2))
    if pooled < 1e-12:
        return float('nan')
    return float((a.mean() - b.mean()) / pooled)

def interpret_d(d):
    if np.isnan(d): return 'N/A'
    ad = abs(d)
    if ad < 0.2:  return 'negligible'
    if ad < 0.5:  return 'small'
    if ad < 0.8:  return 'medium'
    return 'large'

print('=' * 100)
print(f'  COHEN\'S d — Effect Size: Detector vs Baseline @ {TARGET_SPEED} m/s')
print('=' * 100)
print(f'  {"Baseline":<25s} {"Mean Detector":>12s} {"Mean Base":>12s} {"Delta (pp)":>12s}'
      f'  {"d (paired)":>12s} {"d (pooled)":>12s}  Interpretation')
print('  ' + '-' * 95)

cohens_d_results = {}
for key, blist, label in baseline_map:
    if not blist:
        print(f'  {label:<25s}  (tidak ada hasil)')
        continue
    b_arr = per_seed_sorted(blist)
    if len(b_arr) != len(detector_arr):
        print(f'  {label:<25s}  seed mismatch, skip')
        continue
    d_paired = cohens_d(detector_arr, b_arr)
    d_pooled = cohens_d_pooled(detector_arr, b_arr)
    delta    = (detector_arr.mean() - b_arr.mean()) * 100
    interp   = interpret_d(d_paired)
    print(f'  {label:<25s} {detector_arr.mean()*100:>11.2f}% {b_arr.mean()*100:>11.2f}%'
          f' {delta:>+11.2f}pp  {d_paired:>12.4f}  {d_pooled:>12.4f}  {interp}')
    cohens_d_results[key] = dict(
        d_paired=d_paired, d_pooled=d_pooled,
        interpretation=interp, delta_pp=float(delta)
    )

print()
print('  Catatan: d (paired) = mean_diff / std_diff — direkomendasikan untuk paired design.')
print('           d (pooled) = (mean_A - mean_B) / pooled_std — untuk referensi unpaired.')
print('           d < 0.2: negligible | 0.2-0.5: small | 0.5-0.8: medium | > 0.8: large')

import os
with open(os.path.join(BASELINE_DIR, 'cohens_d_baseline.json'), 'w') as f:
    json.dump(cohens_d_results, f, indent=2)
print('\n  Cohen\'s d baseline disimpan ke cohens_d_baseline.json')


  COHEN'S d — Effect Size: Detector vs Baseline @ 25 m/s
  Baseline                  Mean Detector    Mean Base   Delta (pp)    d (paired)   d (pooled)  Interpretation
  -----------------------------------------------------------------------------------------------
  RF+Stat [14]                    98.17%      100.00%       -1.83pp       -1.3633       -1.9280  large
  LSTM-RELU-IO-L1 [39]            98.17%       67.78%      +30.39pp        1.1568        1.5894  large
  BERT WordPiece [25]             98.17%       97.72%       +0.45pp        0.1760        0.1860  negligible

  Catatan: d (paired) = mean_diff / std_diff — direkomendasikan untuk paired design.
           d (pooled) = (mean_A - mean_B) / pooled_std — untuk referensi unpaired.
           d < 0.2: negligible | 0.2-0.5: small | 0.5-0.8: medium | > 0.8: large

  Cohen's d baseline disimpan ke cohens_d_baseline.json


## Cell 17


In [22]:
LIT_VALUES = {'RF_Stat': 94.61, 'LSTM_RELU_IOL1': 95.67, 'BERT_WordPiece': 96.25, 'Detector': 99.34}

METHOD_LABELS = {
    'RF_Stat':         'Random Forest + Stat [14]',
    'LSTM_RELU_IOL1': 'LSTM-RELU-IO-L1 [39]',
    'BERT_WordPiece':  'BERT WordPiece [25]',
    'Detector':           'Detector (Ours)',
}

SEP = chr(9472)
print('=' * 110)
print(f'  Perbandingan Detector vs Baseline @ 25 m/s (REVISI v2)')
print('=' * 110)
print(
    f"| {'Method':<28s} | {'Acc Original Paper (%)':^22s} "
    f"| {'Acc Re-run under Detector Split (%)':^34s} | {'p-value':^14s} | {'Sig':^5s} |"
)
print('|' + SEP*30 + '|' + SEP*24 + '|' + SEP*36 + '|' + SEP*16 + '|' + SEP*7 + '|')

for method in ['RF_Stat', 'LSTM_RELU_IOL1', 'BERT_WordPiece', 'Detector']:
    label   = METHOD_LABELS[method]
    lit_acc = f"{LIT_VALUES.get(method, float('nan')):.2f}"
    rerun   = summary_all.get(method, {}).get('fmt', 'N/A')
    if method == 'Detector':
        pval_s, sig_s = '---', '---'
    else:
        pr = paired_results.get(method, {})
        pv = pr.get('p_value', float('nan'))
        pval_s = f'{pv:.4e}' if not (isinstance(pv, float) and np.isnan(pv)) else 'N/A'
        sig_s  = pr.get('significance', 'N/A')
    print(
        f'| {label:<28s} | {lit_acc:^22s} | {rerun:^34s} | {pval_s:^14s} | {sig_s:^5s} |'
    )

print()
print('Catatan:')
print('  - Semua re-run: 5 seed (42,123,456,789,2024), split episode-based 60/20/20 stratified.')
print('  - create_windows: step_sec=1 (overlapping windows, episode-level split menjamin no leakage).')
print('  - p-value: paired t-test, Detector vs baseline pada split yang identik.')
print('  - *** p<0.001  ** p<0.01  * p<0.05  n.s. = tidak signifikan')
print('  - Lit Detector = nilai original paper; Re-run Detector = rata-rata 5 seeds.')

tabel7 = {}
for method in ['RF_Stat', 'LSTM_RELU_IOL1', 'BERT_WordPiece', 'Detector']:
    tabel7[method] = {
        'lit_acc': LIT_VALUES.get(method),
        'rerun_acc': summary_all.get(method, {}).get('fmt', 'N/A'),
        **paired_results.get(method, {})
    }
with open(os.path.join(BASELINE_DIR, 'tabel_VII.json'), 'w') as f:
    json.dump(tabel7, f, indent=2)
print('Disimpan ke tabel_VII.json')


  Perbandingan Detector vs Baseline @ 25 m/s (REVISI v2)
| Method                       | Acc Original Paper (%) | Acc Re-run under Detector Split (%) |    p-value     |  Sig  |
|──────────────────────────────|────────────────────────|────────────────────────────────────|────────────────|───────|
| Random Forest + Stat [14]    |         94.61          |           100.00 ± 0.00            |   3.8086e-02   |   *   |
| LSTM-RELU-IO-L1 [39]         |         95.67          |           67.78 ± 27.01            |   6.0891e-02   | n.s.  |
| BERT WordPiece [25]          |         96.25          |            97.72 ± 3.12            |   7.1401e-01   | n.s.  |
| Detector (Ours)              |         99.34          |            98.17 ± 1.34            |      ---       |  ---  |

Catatan:
  - Semua re-run: 5 seed (42,123,456,789,2024), split episode-based 60/20/20 stratified.
  - create_windows: step_sec=1 (overlapping windows, episode-level split menjamin no leakage).
  - p-value: paired t-test, 

---
# Output LaTeX

## Cell 18 — LaTeX Rows

In [23]:
def build_bilstm_branch_accurate(window_size, n_feat=5, embedding_dim=32,
                                  name='branch', seed=42):
    """
    Arsitektur IDENTIK dengan build_temporal_lstm yang dipakai saat training:
      Input → BiLSTM(128) → Dropout(0.3) → BiLSTM(64) →
      Dropout(0.3) → Dense(64,relu) → BatchNorm → Dense(32,relu) → Dense(3,softmax)
    """
    tf.keras.utils.set_random_seed(seed)
    inputs    = Input(shape=(window_size, n_feat), name=f'{name}_input')
    x = Bidirectional(LSTM(128, return_sequences=True),  name=f'{name}_bilstm1')(inputs)
    x = Dropout(0.3,                                     name=f'{name}_drop1')(x)
    x = Bidirectional(LSTM(64,  return_sequences=False), name=f'{name}_bilstm2')(x)
    x = Dropout(0.3,                                     name=f'{name}_drop2')(x)
    x = Dense(64,         activation='relu',             name=f'{name}_dense')(x)
    x = BatchNormalization(                              name=f'{name}_bn')(x)
    x = Dense(embedding_dim, activation='relu',          name=f'{name}_emb')(x)
    out = Dense(3,        activation='softmax',          name=f'{name}_out')(x)
    return Model(inputs=inputs, outputs=out, name=f'branch_{name}')


print('=' * 70)
print('  PARAMETER COUNT — Arsitektur Aktual Detector')
print('=' * 70)

WINDOW_ROWS = {1: 60, 5: 300, 15: 900}
total_params = 0
for ws_sec, ws_rows in WINDOW_ROWS.items():
    m = build_bilstm_branch_accurate(ws_rows, n_feat=5, embedding_dim=32,
                                      name=f'w{ws_sec}s')
    trainable = int(np.sum([np.prod(v.shape) for v in m.trainable_weights]))
    total_params += trainable
    print(f'  Branch {ws_sec}s window  : {trainable:>10,} trainable params')
    m.summary(line_length=80, print_fn=lambda x: None)

print(f'  ─────────────────────────────────────────────')
print(f'  Total 3 BiLSTM branches : {total_params:>10,} params')
print(f'  XGBoost meta-learner    : n_estimators=200, max_depth=6 (no gradient params)')
print(f'  ─────────────────────────────────────────────')
print(f'  Total trainable (NN)    : {total_params:>10,} params')
print()
print('Catatan untuk paper:')
print('  - Angka di atas adalah parameter per-branch × 3 branch')
print('  - Window size tidak mempengaruhi param count LSTM')
print('    (LSTM params = f(input_dim, units), bukan f(seq_len))')
print('  - Masukkan angka ini ke tabel arsitektur / complexity analysis di paper')


  PARAMETER COUNT — Arsitektur Aktual Detector
  Branch 1s window  :    312,131 trainable params


  Branch 5s window  :    312,131 trainable params


  Branch 15s window  :    312,131 trainable params


  ─────────────────────────────────────────────
  Total 3 BiLSTM branches :    936,393 params
  XGBoost meta-learner    : n_estimators=200, max_depth=6 (no gradient params)
  ─────────────────────────────────────────────
  Total trainable (NN)    :    936,393 params

Catatan untuk paper:
  - Angka di atas adalah parameter per-branch × 3 branch
  - Window size tidak mempengaruhi param count LSTM
    (LSTM params = f(input_dim, units), bukan f(seq_len))
  - Masukkan angka ini ke tabel arsitektur / complexity analysis di paper


In [24]:
print('=' * 70)
print(f'  PAPER-READY OUTPUT @ {TARGET_SPEED} m/s')
print('=' * 70)

print('\n--- Detector ---')
print(f'{TARGET_SPEED} m/s & '
      f'{np.mean(acc)*100:.2f} ± {np.std(acc,ddof=1)*100:.2f} & '
      f'{np.mean(prec)*100:.2f} ± {np.std(prec,ddof=1)*100:.2f} & '
      f'{np.mean(rec)*100:.2f} ± {np.std(rec,ddof=1)*100:.2f} & '
      f'{np.mean(f1)*100:.2f} ± {np.std(f1,ddof=1)*100:.2f} \\\\')

print('\n--- Ablation ---')
label_map = {
    'xgb_only': 'XGBoost Only',
    'bilstm_1_only': 'BiLSTM Single-Scale (1s)',
    'bilstm_5_only': 'BiLSTM Single-Scale (5s)',
    'bilstm_15_only': 'BiLSTM Single-Scale (15s)',
    'multiscale_bilstm_only': 'Multi-Scale BiLSTM (no kNN/XGB)',
    'multiscale_bilstm_plus_knn': 'Multi-Scale BiLSTM + kNN',
    'full_detector': 'Full Detector',
    'full_amste': 'Full Detector',  # fallback cache lama
}
for k, label in label_map.items():
    if k in ablation_table:
        e = ablation_table[k]
        print(f'{label:<35s} & {e["mean"]:.2f} ± {e["std"]:.2f} \\\\')

print('\n--- k-NN Sensitivity ---')
for k_key in ['k=3', 'k=5', 'k=7', 'k=10']:
    e = k_table[k_key]
    print(f'{k_key} & {e["density_formatted"]} & {e["accuracy_formatted"]} \\\\')

print('\n--- Detector in comparison ---')
print(f'Detector & '
      f'{np.mean(acc)*100:.2f} ± {np.std(acc,ddof=1)*100:.2f} & '
      f'{np.mean(prec)*100:.2f} ± {np.std(prec,ddof=1)*100:.2f} & '
      f'{np.mean(rec)*100:.2f} ± {np.std(rec,ddof=1)*100:.2f} & '
      f'{np.mean(f1)*100:.2f} ± {np.std(f1,ddof=1)*100:.2f} \\\\')

print('\n✅ Done. All summary JSON files are in:', Detector_DIR)
print('\n--- LaTeX ---')
for method in ['RF_Stat', 'LSTM_RELU_IOL1', 'BERT_WordPiece', 'Detector']:
    label   = METHOD_LABELS[method]
    lit_acc = f"{LIT_VALUES.get(method, float('nan')):.2f}"
    rerun   = summary_all.get(method, {}).get('fmt', 'N/A')
    pr      = paired_results.get(method, {})
    pval_s  = f"{pr.get('p_value', float('nan')):.4e}" if method != 'Detector' else '—'
    sig_s   = pr.get('significance', '—') if method != 'Detector' else '—'
    print(f'{label} & {lit_acc} & {rerun} & {pval_s} & {sig_s} \\\\')


  PAPER-READY OUTPUT @ 25 m/s

--- Detector ---
25 m/s & 98.17 ± 1.34 & 97.13 ± 2.67 & 98.42 ± 1.26 & 97.66 ± 1.89 \\

--- Ablation ---
XGBoost Only                        & 100.00 ± 0.00 \\
BiLSTM Single-Scale (1s)            & 91.41 ± 1.45 \\
BiLSTM Single-Scale (5s)            & 96.03 ± 0.87 \\
BiLSTM Single-Scale (15s)           & 99.33 ± 0.50 \\
Multi-Scale BiLSTM (no kNN/XGB)     & 98.75 ± 1.39 \\
Multi-Scale BiLSTM + kNN            & 98.75 ± 1.39 \\
Full Detector                       & 98.17 ± 1.34 \\

--- k-NN Sensitivity ---
k=3 & 0.0219 ± 0.0089 & 98.17 ± 1.34 \\
k=5 & 0.0365 ± 0.0148 & 98.17 ± 1.34 \\
k=7 & 0.0511 ± 0.0207 & 98.17 ± 1.34 \\
k=10 & 0.0730 ± 0.0295 & 98.17 ± 1.34 \\

--- Detector in comparison ---
Detector & 98.17 ± 1.34 & 97.13 ± 2.67 & 98.42 ± 1.26 & 97.66 ± 1.89 \\

✅ Done. All summary JSON files are in: /content/drive/MyDrive/Detector_Results/seed_runs_25ms

--- LaTeX ---
Random Forest + Stat [14] & 94.61 & 100.00 ± 0.00 & 3.8086e-02 & * \\
LSTM-RELU-IO-L

## Cell 18b — LaTeX Output: Cohen's d + Ablation Stats

In [25]:
print('=' * 70)
print(f'  FINAL (dengan Cohen\'s d) @ {TARGET_SPEED} m/s')
print('=' * 70)
print()
print('Format: Method & Lit & Re-run & p-value & Sig & d & Interpretation')
print()

for method in ['RF_Stat', 'LSTM_RELU_IOL1', 'BERT_WordPiece', 'Detector']:
    label   = METHOD_LABELS[method]
    lit_acc = f"{LIT_VALUES.get(method, float('nan')):.2f}"
    rerun   = summary_all.get(method, {}).get('fmt', 'N/A')
    if method == 'Detector':
        pval_s = '---'; sig_s = '---'; d_s = '---'; interp_s = '---'
    else:
        pr     = paired_results.get(method, {})
        cd     = cohens_d_results.get(method, {})
        pval_s = f"{pr.get('p_value', float('nan')):.4e}"
        sig_s  = pr.get('significance', 'N/A')
        d_val  = cd.get('d_paired', float('nan'))
        d_s    = f'{d_val:.3f}' if not np.isnan(d_val) else 'N/A'
        interp_s = cd.get('interpretation', 'N/A')
    print(f'{label} & {lit_acc} & {rerun} & {pval_s} & {sig_s} & {d_s} & {interp_s} \\\\')

print()
print('--- Ablation + Statistik ---')
print('Format: Component & Mean(%) & Delta(pp) & p-value & Sig & d & Interp')
print()
for abl_key, abl_label in [
    ('xgb_only',                 'XGBoost Only'),
    ('bilstm_1_only',            'BiLSTM Single-Scale (1s)'),
    ('bilstm_5_only',            'BiLSTM Single-Scale (5s)'),
    ('bilstm_15_only',           'BiLSTM Single-Scale (15s)'),
    ('multiscale_bilstm_only',   'Multi-Scale BiLSTM (no kNN/XGB)'),
    ('multiscale_bilstm_plus_knn','Multi-Scale BiLSTM + kNN'),
    ('full_detector',               'Full Detector'),
]:
    if abl_key == 'full_detector':
        e = ablation_table.get('full_detector', {})
        print(f'{abl_label:<35s} & {e.get("mean", 0):.2f} ± {e.get("std", 0):.2f} & --- & --- & --- & --- & --- \\\\')
        continue
    e   = ablation_table.get(abl_key, {})
    st  = ablation_stat_results.get(abl_key, {})
    pv  = st.get('p_value', float('nan'))
    pv_s = f'{pv:.4e}' if not np.isnan(pv) else 'N/A'
    d_v  = st.get('cohens_d', float('nan'))
    d_s  = f'{d_v:.3f}' if not np.isnan(d_v) else 'N/A'
    print(f'{abl_label:<35s} & {e.get("mean",0):.2f} ± {e.get("std",0):.2f}'
          f' & {st.get("delta_pp",0):>+.2f}pp & {pv_s} & {st.get("significance","N/A")}'
          f' & {d_s} & {st.get("interpretation","N/A")} \\\\')


  FINAL (dengan Cohen's d) @ 25 m/s

Format: Method & Lit & Re-run & p-value & Sig & d & Interpretation

Random Forest + Stat [14] & 94.61 & 100.00 ± 0.00 & 3.8086e-02 & * & -1.363 & large \\
LSTM-RELU-IO-L1 [39] & 95.67 & 67.78 ± 27.01 & 6.0891e-02 & n.s. & 1.157 & large \\
BERT WordPiece [25] & 96.25 & 97.72 ± 3.12 & 7.1401e-01 & n.s. & 0.176 & negligible \\
Detector (Ours) & 99.34 & 98.17 ± 1.34 & --- & --- & --- & --- \\

--- Ablation + Statistik ---
Format: Component & Mean(%) & Delta(pp) & p-value & Sig & d & Interp

XGBoost Only                        & 100.00 ± 0.00 & -1.83pp & 3.8086e-02 & * & -1.363 & large \\
BiLSTM Single-Scale (1s)            & 91.41 ± 1.45 & +6.76pp & 7.1900e-04 & *** & 4.196 & large \\
BiLSTM Single-Scale (5s)            & 96.03 ± 0.87 & +2.14pp & 1.9631e-02 & * & 1.685 & large \\
BiLSTM Single-Scale (15s)           & 99.33 ± 0.50 & -1.16pp & 8.0502e-02 & n.s. & -1.041 & large \\
Multi-Scale BiLSTM (no kNN/XGB)     & 98.75 ± 1.39 & -0.58pp & 3.7390e-01 &

In [26]:
import json, os

for seed in SEEDS:
    fp = os.path.join(Detector_DIR, f'seed_{seed}_result.json')
    if os.path.exists(fp):
        with open(fp) as f:
            r = json.load(f)
        print(f"Seed {seed}: accuracy = {r['accuracy']*100:.2f}%  | keys: {list(r.keys())[:5]}")
    else:
        print(f"Seed {seed}: ❌ FILE TIDAK ADA")

Seed 42: accuracy = 97.13%  | keys: ['seed', 'target_speed', 'accuracy', 'precision_macro', 'recall_macro']
Seed 123: accuracy = 99.20%  | keys: ['seed', 'target_speed', 'accuracy', 'precision_macro', 'recall_macro']
Seed 456: accuracy = 97.09%  | keys: ['seed', 'target_speed', 'accuracy', 'precision_macro', 'recall_macro']
Seed 789: accuracy = 97.44%  | keys: ['seed', 'target_speed', 'accuracy', 'precision_macro', 'recall_macro']
Seed 2024: accuracy = 100.00%  | keys: ['seed', 'target_speed', 'accuracy', 'precision_macro', 'recall_macro']


# Detector_Complexity_Benchmark

## Cell 1 — Setup & Import

In [27]:
import os, sys, time, json, gc, tracemalloc, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
assert len(gpus) > 0, "❌ Aktifkan GPU runtime: Runtime → Change runtime type → T4 GPU"

for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

gpu_name = tf.test.gpu_device_name()
print(f"✅ TF version   : {tf.__version__}")
print(f"✅ PyTorch ver  : {torch.__version__}")
print(f"✅ CUDA avail   : {torch.cuda.is_available()}")
print(f"✅ GPU device   : {gpu_name}")
print(f"✅ GPU (torch)  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

BENCHMARK_OUT = '/content/drive/MyDrive/Detector_Results/complexity_benchmark_v3_25m/s.json'
os.makedirs(os.path.dirname(BENCHMARK_OUT), exist_ok=True)
print(f"\n✅ Output JSON  : {BENCHMARK_OUT}")

✅ TF version   : 2.20.0
✅ PyTorch ver  : 2.11.0+cu128
✅ CUDA avail   : True
✅ GPU device   : /device:GPU:0
✅ GPU (torch)  : Tesla T4
Tesla T4, 15360 MiB, 580.82.07

✅ Output JSON  : /content/drive/MyDrive/Detector_Results/complexity_benchmark_v3_25m/s.json


## Cell 2 — Mount Drive

In [28]:
try:
    from google.colab import drive
    import subprocess, os

    mount_point = '/content/drive'

    # Cek apakah sudah ter-mount dengan benar (ada MyDrive)
    if os.path.isdir(os.path.join(mount_point, 'MyDrive')):
        print('✅ Drive sudah ter-mount sebelumnya — skip remount.')
    else:
        if os.path.isdir(mount_point) and os.listdir(mount_point):
            print('⚠️ /content/drive berisi file tapi bukan mount valid. Membersihkan...')
            subprocess.run(['fusermount', '-uz', mount_point],
                           capture_output=True)
            import time; time.sleep(1)
            for item in os.listdir(mount_point):
                item_path = os.path.join(mount_point, item)
                if os.path.isdir(item_path):
                    import shutil; shutil.rmtree(item_path)
                else:
                    os.remove(item_path)
            print('  Direktori dibersihkan.')

        drive.mount(mount_point, force_remount=True)
        print('✅ Drive mounted.')

    if os.path.isdir(os.path.join(mount_point, 'MyDrive')):
        print('✅ Verifikasi: MyDrive tersedia.')
    else:
        print('❌ MyDrive tidak ditemukan — periksa akses Drive.')

except ImportError:
    print('⚠️ Bukan environment Colab — skip mount.')
except Exception as e:
    print(f'❌ Error: {e}')
    print('   Coba: Runtime → Disconnect and delete runtime, lalu run ulang.')


✅ Drive sudah ter-mount sebelumnya — skip remount.
✅ Verifikasi: MyDrive tersedia.


## Cell 3 — Helper Functions

**Penting:** Dua backend GPU sync:
- TensorFlow models: `tf_gpu_sync()` via eager evaluation
- PyTorch models: `torch.cuda.synchronize()` — lebih presisi

In [29]:
def tf_gpu_sync():
    """Force TF GPU sync via eager op."""
    if tf.config.list_physical_devices('GPU'):
        _ = tf.constant(0.0).numpy()

def torch_gpu_sync():
    """Force PyTorch CUDA sync."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def measure_latency_tf(forward_fn, sample_input, n_warmup=100, n_runs=1000, name='model'):
    """Latency untuk TF/Keras model."""
    print(f'  [{name}] TF warm-up {n_warmup}...', end=' ')
    for _ in range(n_warmup): _ = forward_fn(sample_input)
    tf_gpu_sync(); print('done.')

    print(f'  [{name}] TF timed {n_runs}...', end=' ')
    lats = []
    for _ in range(n_runs):
        tf_gpu_sync()
        t0 = time.perf_counter()
        _ = forward_fn(sample_input)
        tf_gpu_sync()
        lats.append((time.perf_counter() - t0) * 1000)
    print('done.')
    arr = np.array(lats)
    return dict(mean_ms=float(arr.mean()), std_ms=float(arr.std()),
                p50_ms=float(np.percentile(arr,50)), p95_ms=float(np.percentile(arr,95)),
                p99_ms=float(np.percentile(arr,99)), min_ms=float(arr.min()), max_ms=float(arr.max()),
                n_warmup=n_warmup, n_runs=n_runs)

def measure_latency_torch(forward_fn, sample_input, n_warmup=100, n_runs=1000, name='model'):
    """Latency untuk PyTorch model — pakai torch.cuda.synchronize()."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if isinstance(sample_input, dict):
        sample_input = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                        for k, v in sample_input.items()}
    elif isinstance(sample_input, torch.Tensor):
        sample_input = sample_input.to(device)

    print(f'  [{name}] Torch warm-up {n_warmup}...', end=' ')
    with torch.no_grad():
        for _ in range(n_warmup): _ = forward_fn(sample_input)
    torch_gpu_sync(); print('done.')

    print(f'  [{name}] Torch timed {n_runs}...', end=' ')
    lats = []
    with torch.no_grad():
        for _ in range(n_runs):
            torch_gpu_sync()
            t0 = time.perf_counter()
            _ = forward_fn(sample_input)
            torch_gpu_sync()
            lats.append((time.perf_counter() - t0) * 1000)
    print('done.')
    arr = np.array(lats)
    return dict(mean_ms=float(arr.mean()), std_ms=float(arr.std()),
                p50_ms=float(np.percentile(arr,50)), p95_ms=float(np.percentile(arr,95)),
                p99_ms=float(np.percentile(arr,99)), min_ms=float(arr.min()), max_ms=float(arr.max()),
                n_warmup=n_warmup, n_runs=n_runs)

def measure_peak_memory_tf(forward_fn, sample_input, name='model'):
    """Peak memory untuk TF model."""
    gc.collect()
    if tf.config.list_physical_devices('GPU'):
        try: tf.config.experimental.reset_memory_stats('GPU:0')
        except: pass
    tracemalloc.start()
    _ = forward_fn(sample_input)
    tf_gpu_sync()
    try:
        mi = tf.config.experimental.get_memory_info('GPU:0')
        gpu_peak = mi.get('peak', 0) / 1024**2
    except: gpu_peak = 0.0
    _, cpu_peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return dict(gpu_peak_mb=float(gpu_peak), cpu_peak_mb=float(cpu_peak/1024**2))

def measure_peak_memory_torch(forward_fn, sample_input, name='model'):
    """Peak memory untuk PyTorch model."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if isinstance(sample_input, dict):
        sample_input = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                        for k, v in sample_input.items()}
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    tracemalloc.start()
    with torch.no_grad(): _ = forward_fn(sample_input)
    torch_gpu_sync()
    gpu_peak = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
    _, cpu_peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return dict(gpu_peak_mb=float(gpu_peak), cpu_peak_mb=float(cpu_peak/1024**2))

def count_keras_params(model):
    return int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

def count_torch_params(model):
    return sum(p.numel() for p in model.parameters())

def get_xgb_size_mb(xgb_model, path='/tmp/_xgb_probe.json'):
    xgb_model.save_model(path)
    sz = os.path.getsize(path) / 1024**2
    os.remove(path); return sz

def _extract_one_tab(x_padded_single):
    x = x_padded_single[0]
    m = ~(x == 0).all(axis=1)
    xv = x[m] if m.sum() > 0 else x
    f = []
    for c in range(xv.shape[1]):
        v = xv[:, c]
        f.extend([np.mean(v), np.std(v), np.min(v), np.max(v),
                  np.median(v), np.percentile(v, 25), np.percentile(v, 75)])
    return np.array(f, dtype=np.float32).reshape(1, -1)

print('✅ Helper functions defined (TF + PyTorch backends)')


✅ Helper functions defined (TF + PyTorch backends)


## Cell 4 — Detect Mode (INLINE vs STANDALONE)

In [30]:
_INLINE_MODE = False
try:
    _ = embedding_models
    _ = meta
    _ = xgb
    _ = processed
    _ = sc_meta
    _INLINE_MODE = True
    print('✅ INLINE MODE — model objek dari notebook Detector terdeteksi.')
    print('   embedding_models, meta, xgb, processed, sc_meta siap dipakai.')
except NameError:
    print('ℹ STANDALONE MODE — akan retrain seed 42.')

BENCHMARK_SEED = 42
TARGET_SPEED_BENCH = 25
DATASET_PATH_BENCH = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
SPEED_LOW  = 23
SPEED_HIGH = 27

import os, glob

if not os.path.exists(DATASET_PATH_BENCH):
    print(f'\n⚠️  File tidak ditemukan di: {DATASET_PATH_BENCH}')
    print('   Mencari file Dataset*.csv di Google Drive...')
    candidates = glob.glob('/content/drive/MyDrive/**/Dataset*.csv', recursive=True)
    if candidates:
        print(f'   Ditemukan {len(candidates)} kandidat:')
        for i, c in enumerate(candidates):
            print(f'     [{i}] {c}')
        print()
        print('   ➡️  Copy path yang sesuai ke DATASET_PATH_BENCH di atas, lalu run ulang cell ini.')
    else:
        print('   ❌ Tidak ada file Dataset*.csv di Drive.')
        print('   Pastikan file sudah diupload ke Google Drive.')
else:
    sz = os.path.getsize(DATASET_PATH_BENCH) / 1024**2
    print(f'\n✅ Dataset ditemukan: {DATASET_PATH_BENCH}')
    print(f'   File size: {sz:.1f} MB')
    print(f'   Target speed: {TARGET_SPEED_BENCH} m/s  (filter: {SPEED_LOW}–{SPEED_HIGH} m/s)')


ℹ STANDALONE MODE — akan retrain seed 42.

✅ Dataset ditemukan: /content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv
   File size: 0.2 MB
   Target speed: 25 m/s  (filter: 23–27 m/s)


STANDALONE: Retrain Seed 42 (Skip kalau INLINE MODE)

In [31]:
if not _INLINE_MODE:
    import random
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import (Input, Bidirectional, LSTM, Dense,
                                          Dropout, BatchNormalization)
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from sklearn.preprocessing import StandardScaler, RobustScaler
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.utils.class_weight import compute_class_weight
    from sklearn.metrics import accuracy_score

    FEATURES       = ['SNR', 'RSSI', 'PDR', 'Speed', 'Relative_Speed']
    WINDOW_SIZES   = [1, 5, 15]
    EPISODE_LENGTH = 60
    K_NEIGHBORS    = 5

    print('Loading dataset...')
    df = pd.read_csv(DATASET_PATH_BENCH)
    df = df[(df['Speed'] >= SPEED_LOW) & (df['Speed'] <= SPEED_HIGH)].copy()
    df['label'] = df['Scenario'].astype(int) - 1
    df.dropna(subset=['label'], inplace=True)
    df = df.drop_duplicates().reset_index(drop=True)
    if 'pseudo_run_id' not in df.columns:
        df['time_bucket']   = (df['Time'] // EPISODE_LENGTH).astype(int)
        df['speed_rounded'] = df['Speed'].round().astype(int)
        df['pseudo_run_id'] = (df['speed_rounded'].astype(str) + '_' +
                               df['Scenario'].astype(str) + '_' +
                               df['time_bucket'].astype(str))
    df_speed = df.copy()
    print(f'Dataset: {len(df_speed)} rows, {df_speed["pseudo_run_id"].nunique()} episodes')

    np.random.seed(BENCHMARK_SEED); random.seed(BENCHMARK_SEED)
    tf.random.set_seed(BENCHMARK_SEED)
    os.environ['PYTHONHASHSEED'] = str(BENCHMARK_SEED)

    rng = np.random.RandomState(BENCHMARK_SEED)
    ep_to_label = df_speed.groupby('pseudo_run_id')['label'].agg(lambda x: x.iloc[0]).to_dict()
    eps_by_class = {}
    for ep, lbl in ep_to_label.items():
        if pd.isna(lbl): continue
        eps_by_class.setdefault(int(lbl), []).append(ep)
    train_eps, val_eps, test_eps = [], [], []
    for lbl, eps in eps_by_class.items():
        eps = np.array(eps); rng.shuffle(eps); n = len(eps)
        n_train = int(round(0.6 * n)); n_val = int(round(0.2 * n))
        if n >= 3:
            n_train = max(1, min(n - 2, n_train))
            n_val   = max(1, min(n - n_train - 1, n_val))
        train_eps += eps[:n_train].tolist()
        val_eps   += eps[n_train:n_train+n_val].tolist()
        test_eps  += eps[n_train+n_val:].tolist()
    train_df = df_speed[df_speed['pseudo_run_id'].isin(train_eps)].copy()
    val_df   = df_speed[df_speed['pseudo_run_id'].isin(val_eps)].copy()
    test_df  = df_speed[df_speed['pseudo_run_id'].isin(test_eps)].copy()

    def create_windows(df, window_sec):
        X_list, y_list = [], []
        for _, group in df.groupby('pseudo_run_id'):
            group = group.sort_values('Time').reset_index(drop=True)
            times  = group['Time'].values; X_raw = group[FEATURES].values; y_raw = group['label'].values
            t0 = times[0]
            while t0 + window_sec <= times[-1]:
                mask = (times >= t0) & (times < t0 + window_sec)
                if mask.sum() > 0:
                    X_list.append(X_raw[mask])
                    y_list.append(np.bincount(y_raw[mask].astype(int)).argmax())
                t0 += window_sec
        return X_list, np.array(y_list)

    def pad_normalize(Xtr_l, Xv_l, Xte_l):
        all_d = Xtr_l + Xv_l + Xte_l
        if not all_d: return np.zeros((0,0,len(FEATURES))), np.zeros((0,0,len(FEATURES))), np.zeros((0,0,len(FEATURES))), 0
        max_len = max(len(x) for x in all_d); n_feat = all_d[0].shape[1]
        def pad(L):
            if not L: return np.zeros((0, max_len, n_feat), dtype=np.float32)
            out = np.zeros((len(L), max_len, n_feat), dtype=np.float32)
            for i, x in enumerate(L): sl = min(len(x), max_len); out[i, :sl] = x[:sl]
            return out
        Xtr, Xv, Xte = pad(Xtr_l), pad(Xv_l), pad(Xte_l)
        sc = StandardScaler()
        Xtr_s = sc.fit_transform(Xtr.reshape(-1,n_feat)).reshape(Xtr.shape) if Xtr.shape[0]>0 else Xtr
        Xv_s  = sc.transform(Xv.reshape(-1,n_feat)).reshape(Xv.shape)   if Xv.shape[0]>0  else Xv
        Xte_s = sc.transform(Xte.reshape(-1,n_feat)).reshape(Xte.shape) if Xte.shape[0]>0 else Xte
        return Xtr_s, Xv_s, Xte_s, max_len

    processed = {}
    for ws in WINDOW_SIZES:
        Xtr, ytr = create_windows(train_df, ws)
        Xv,  yv  = create_windows(val_df,   ws)
        Xte, yte = create_windows(test_df,  ws)
        Xtr_s, Xv_s, Xte_s, ml = pad_normalize(Xtr, Xv, Xte)
        processed[f'{ws}s'] = dict(X_train=Xtr_s, X_val=Xv_s, X_test=Xte_s,
                                    y_train=ytr, y_val=yv, y_test=yte, max_len=ml)
        print(f'  Window {ws}s: train={len(ytr)}, val={len(yv)}, test={len(yte)}, max_len={ml}')

    def build_temporal_lstm(input_shape, embedding_dim=32, name='lstm', seed=42):
        tf.keras.utils.set_random_seed(seed)
        inp = Input(shape=input_shape, name=f'{name}_input')
        x   = Bidirectional(LSTM(128, return_sequences=True), name=f'{name}_bilstm1')(inp)
        x   = Dropout(0.3)(x)
        x   = Bidirectional(LSTM(64,  return_sequences=False), name=f'{name}_bilstm2')(x)
        x   = Dropout(0.3)(x)
        x   = Dense(64, activation='relu', name=f'{name}_dense')(x)
        x   = BatchNormalization()(x)
        emb = Dense(embedding_dim, activation='relu', name=f'{name}_emb')(x)
        out = Dense(3, activation='softmax', name=f'{name}_out')(emb)
        return Model(inp, out, name=f'{name}_full'), Model(inp, emb, name=f'{name}_e')

    embedding_models = {}
    print('\nTraining BiLSTM × 3 (seed=42)...')
    for ws in WINDOW_SIZES:
        d = processed[f'{ws}s']
        full, emb = build_temporal_lstm((d['max_len'], len(FEATURES)), 32,
                                         name=f'lstm_{ws}s_{BENCHMARK_SEED}', seed=BENCHMARK_SEED)
        full.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        cb = [EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
              ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
        full.fit(d['X_train'], d['y_train'], validation_data=(d['X_val'], d['y_val']),
                 epochs=100, batch_size=32, callbacks=cb, verbose=0)
        embedding_models[f'{ws}s'] = emb
        _, acc = full.evaluate(d['X_test'], d['y_test'], verbose=0)
        print(f'  BiLSTM {ws}s acc: {acc:.4f}')

    def compute_spatial_features(X_padded, k=5):
        if X_padded.shape[0] == 0: return np.zeros((0, len(FEATURES)*2+2))
        X_mean = X_padded.mean(axis=1); sim = cosine_similarity(X_mean); np.fill_diagonal(sim, 0.0)
        out = []
        for i in range(X_padded.shape[0]):
            s = sim[i]; k_act = min(k, len(s)-1)
            if k_act > 0:
                idx = np.argsort(s)[-k_act:]; ns = s[idx]
                nm = X_mean[idx].mean(axis=0); nstd = X_mean[idx].std(axis=0)
                feat = np.concatenate([nm, nstd, [ns.mean()], [ns.std()]])
            else: feat = np.zeros(len(FEATURES)*2+2)
            out.append(feat)
        return np.array(out)

    def extract_tabular_features(X_padded):
        if X_padded.shape[0] == 0: return np.zeros((0, len(FEATURES)*7))
        out = []
        for x in X_padded:
            m = ~(x == 0).all(axis=1); xv = x[m] if m.sum() > 0 else x
            f = []
            for c in range(xv.shape[1]):
                v = xv[:,c]; f.extend([np.mean(v), np.std(v), np.min(v), np.max(v),
                                       np.median(v), np.percentile(v,25), np.percentile(v,75)])
            out.append(f)
        return np.array(out)
    print('\nTraining XGBoost branch...')
    Xt_tr = extract_tabular_features(processed['15s']['X_train'])
    Xt_v  = extract_tabular_features(processed['15s']['X_val'])
    y_tr_ref = processed['15s']['y_train']; y_v_ref = processed['15s']['y_val']
    _xgb_has_val = Xt_v.shape[0] > 0 and y_v_ref.shape[0] > 0
    xgb = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=200,
                         max_depth=5, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, random_state=BENCHMARK_SEED,
                         eval_metric='mlogloss',
                         early_stopping_rounds=20 if _xgb_has_val else None)
    if _xgb_has_val:
        xgb.fit(Xt_tr, y_tr_ref, eval_set=[(Xt_v, y_v_ref)], verbose=False)
    else:
        xgb.fit(Xt_tr, y_tr_ref, verbose=False)
        print('  (val kosong — fit tanpa early stopping)')
    print(f'  XGBoost-only trained.')
    print('\nTraining meta-classifier...')

    def safe_predict(model, X, embed_dim=32):
        if X.shape[0] == 0:
            return np.zeros((0, embed_dim), dtype=np.float32)
        return model.predict(X, verbose=0)

    e_all = {f'{ws}s': {sp: safe_predict(embedding_models[f'{ws}s'],
                                          processed[f'{ws}s'][f'X_{sp}'])
                         for sp in ['train','val','test']} for ws in WINDOW_SIZES}
    Xt_te = extract_tabular_features(processed['15s']['X_test'])
    sp_tr = compute_spatial_features(processed['15s']['X_train'], K_NEIGHBORS)
    sp_v  = compute_spatial_features(processed['15s']['X_val'],   K_NEIGHBORS)
    p_tr = xgb.predict_proba(Xt_tr); p_v = xgb.predict_proba(Xt_v)
    n_min = min(e_all['1s']['train'].shape[0], e_all['5s']['train'].shape[0],
            e_all['15s']['train'].shape[0], sp_tr.shape[0], p_tr.shape[0])
    Z_tr = np.hstack([e_all['1s']['train'][:n_min], e_all['5s']['train'][:n_min],
                  e_all['15s']['train'][:n_min], sp_tr[:n_min], p_tr[:n_min]])

    n_v = min(e_all['1s']['val'].shape[0], e_all['5s']['val'].shape[0],
          e_all['15s']['val'].shape[0], sp_v.shape[0], p_v.shape[0])
    Z_v = np.hstack([e_all['1s']['val'][:n_v], e_all['5s']['val'][:n_v],
                 e_all['15s']['val'][:n_v], sp_v[:n_v], p_v[:n_v]])

    y_tr_a = y_tr_ref[:Z_tr.shape[0]]; y_v_a = y_v_ref[:Z_v.shape[0]]
    sc_meta = RobustScaler()
    Z_tr_s = sc_meta.fit_transform(Z_tr)
    Z_v_s  = sc_meta.transform(Z_v) if Z_v.shape[0] > 0 else np.zeros((0, Z_v.shape[1]), dtype=np.float32)

    cw = compute_class_weight('balanced', classes=np.unique(y_tr_a), y=y_tr_a)
    sw = np.array([cw[y] for y in y_tr_a])
    _meta_has_val = Z_v_s.shape[0] > 0 and y_v_a.shape[0] > 0
    meta = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300, max_depth=4,
                          learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
                          random_state=BENCHMARK_SEED, eval_metric='mlogloss',
                          early_stopping_rounds=20 if _meta_has_val else None)
    if _meta_has_val:
        meta.fit(Z_tr_s, y_tr_a, eval_set=[(Z_v_s, y_v_a)],
                 sample_weight=sw, verbose=False)
    else:
        meta.fit(Z_tr_s, y_tr_a, sample_weight=sw, verbose=False)
        print('  (val kosong — fit tanpa early stopping)')
    print('\n✅ Semua model siap untuk benchmark.')

Loading dataset...
Dataset: 1195 rows, 12 episodes
  Window 1s: train=113, val=7, test=110, max_len=10
  Window 5s: train=23, val=2, test=20, max_len=31
  Window 15s: train=6, val=0, test=6, max_len=82

Training BiLSTM × 3 (seed=42)...
  BiLSTM 1s acc: 0.9273
  BiLSTM 5s acc: 0.8500
  BiLSTM 15s acc: 0.6667

Training XGBoost branch...
  (val kosong — fit tanpa early stopping)
  XGBoost-only trained.

Training meta-classifier...


  (val kosong — fit tanpa early stopping)

✅ Semua model siap untuk benchmark.


In [32]:
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight

def safe_predict(model, X, embed_dim=32):
    if X.shape[0] == 0:
        return np.zeros((0, embed_dim), dtype=np.float32)
    return model.predict(X, verbose=0)

e_all = {f'{ws}s': {sp: safe_predict(embedding_models[f'{ws}s'],
                                      processed[f'{ws}s'][f'X_{sp}'])
                     for sp in ['train','val','test']} for ws in [1,5,15]}

sp_tr = compute_spatial_features(processed['15s']['X_train'], 5)
p_tr  = xgb.predict_proba(Xt_tr)

n_tr = min(e_all['1s']['train'].shape[0], e_all['5s']['train'].shape[0],
           e_all['15s']['train'].shape[0], sp_tr.shape[0], p_tr.shape[0])
print(f'n_tr after align: {n_tr}')

Z_tr  = np.hstack([e_all['1s']['train'][:n_tr], e_all['5s']['train'][:n_tr],
                   e_all['15s']['train'][:n_tr], sp_tr[:n_tr], p_tr[:n_tr]])
y_tr_a = y_tr_ref[:n_tr]

sc_meta = RobustScaler()
Z_tr_s  = sc_meta.fit_transform(Z_tr)

cw = compute_class_weight('balanced', classes=np.unique(y_tr_a), y=y_tr_a)
sw = np.array([cw[y] for y in y_tr_a])

meta = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300,
                      max_depth=4, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=3, gamma=0.1,
                      reg_alpha=0.1, reg_lambda=1.0, random_state=42,
                      eval_metric='mlogloss')
meta.fit(Z_tr_s, y_tr_a, sample_weight=sw, verbose=False)
print('✅ meta-classifier trained. Lanjut ke Cell 5.')

n_tr after align: 6
✅ meta-classifier trained. Lanjut ke Cell 5.


## Cell 5 — Siapkan Sample Inputs (batch_size=1)

In [33]:
sample_X_1s  = processed['1s']['X_test'][0:1]
sample_X_5s  = processed['5s']['X_test'][0:1]
sample_X_15s = processed['15s']['X_test'][0:1]
sample_tab   = _extract_one_tab(sample_X_15s)

print(f'Sample shapes:')
print(f'  X_1s  : {sample_X_1s.shape}')
print(f'  X_5s  : {sample_X_5s.shape}')
print(f'  X_15s : {sample_X_15s.shape}')
print(f'  tabular: {sample_tab.shape}')
print(f'\nWindow sizes: 1s={sample_X_1s.shape[1]} steps, 5s={sample_X_5s.shape[1]} steps, 15s={sample_X_15s.shape[1]} steps')


Sample shapes:
  X_1s  : (1, 10, 5)
  X_5s  : (1, 31, 5)
  X_15s : (1, 82, 5)
  tabular: (1, 35)

Window sizes: 1s=10 steps, 5s=31 steps, 15s=82 steps


## Cell 6 — Benchmark: XGBoost-only (OBU tier)

In [34]:
print('='*70)
print('BENCHMARK: XGBoost-only (OBU tier)')
print('='*70)

def xgb_pure(x_tab): return xgb.predict_proba(x_tab)
def xgb_e2e(x_raw):  return xgb.predict_proba(_extract_one_tab(x_raw))

print('\n[a] Pure prediction:')
lat_xgb_pure = measure_latency_tf(xgb_pure, sample_tab, n_warmup=100, n_runs=1000, name='xgb_pure')
for k, v in lat_xgb_pure.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

print('\n[b] End-to-end (feat extraction + XGBoost):')
lat_xgb_e2e = measure_latency_tf(xgb_e2e, sample_X_15s, n_warmup=100, n_runs=1000, name='xgb_e2e')
for k, v in lat_xgb_e2e.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

mem_xgb = measure_peak_memory_tf(xgb_pure, sample_tab, name='xgb')
print(f'\nMemory:')
print(f'  GPU peak: {mem_xgb["gpu_peak_mb"]:.4f} MB  (expected ~0; XGBoost is CPU-bound)')
print(f'  CPU peak: {mem_xgb["cpu_peak_mb"]:.4f} MB')

xgb_size = get_xgb_size_mb(xgb)
print(f'  Model file size: {xgb_size:.4f} MB')

RESULT_XGB = dict(config_name='XGBoost-only', target_tier='OBU',
                  trainable_params_nn=0, xgb_n_estimators=200, xgb_max_depth=5,
                  latency_pure_ms=lat_xgb_pure, latency_e2e_ms=lat_xgb_e2e,
                  memory_mb=mem_xgb, model_file_size_mb=xgb_size)
print('\n✅ XGBoost benchmark done.')


BENCHMARK: XGBoost-only (OBU tier)

[a] Pure prediction:
  [xgb_pure] TF warm-up 100... done.
  [xgb_pure] TF timed 1000... done.
  mean_ms        : 0.6063 ms
  std_ms         : 0.3364 ms
  p50_ms         : 0.5546 ms
  p95_ms         : 0.6696 ms
  p99_ms         : 2.1924 ms
  min_ms         : 0.5054 ms
  max_ms         : 6.2401 ms

[b] End-to-end (feat extraction + XGBoost):
  [xgb_e2e] TF warm-up 100... done.
  [xgb_e2e] TF timed 1000... done.
  mean_ms        : 2.6531 ms
  std_ms         : 0.5947 ms
  p50_ms         : 2.5369 ms
  p95_ms         : 3.0038 ms
  p99_ms         : 6.0392 ms
  min_ms         : 2.2389 ms
  max_ms         : 9.4578 ms

Memory:
  GPU peak: 76.4031 MB  (expected ~0; XGBoost is CPU-bound)
  CPU peak: 0.0099 MB
  Model file size: 0.2695 MB

✅ XGBoost benchmark done.


## Cell 7 — Benchmark: Single-Scale BiLSTM-15s (RSU tier)

In [35]:
from tensorflow.keras.layers import Dense as KDense
from tensorflow.keras.models import Model as KModel

print('='*70)
print('BENCHMARK: Single-Scale BiLSTM-15s (RSU tier)')
print('='*70)

emb_15s = embedding_models['15s']
_inp = emb_15s.input
_out_emb = emb_15s.output
_softmax = KDense(3, activation='softmax', name='bench_softmax')(_out_emb)
bilstm_15s_full = KModel(_inp, _softmax, name='bilstm_15s_bench')

n_params_bilstm = count_keras_params(bilstm_15s_full)
print(f'BiLSTM-15s trainable params: {n_params_bilstm:,}')
print(f'  (BiLSTM branch embedding model: {count_keras_params(emb_15s):,})')

def bilstm_15s_fwd(x): return bilstm_15s_full(x, training=False)

lat_bilstm = measure_latency_tf(bilstm_15s_fwd, sample_X_15s, n_warmup=100, n_runs=1000, name='bilstm_15s')
print('\nLatency:')
for k, v in lat_bilstm.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

mem_bilstm = measure_peak_memory_tf(bilstm_15s_fwd, sample_X_15s, name='bilstm_15s')
print(f'\nMemory:')
print(f'  GPU peak: {mem_bilstm["gpu_peak_mb"]:.4f} MB')
print(f'  CPU peak: {mem_bilstm["cpu_peak_mb"]:.4f} MB')

RESULT_BILSTM = dict(config_name='BiLSTM-15s', target_tier='RSU',
                     trainable_params_nn=n_params_bilstm,
                     latency_ms=lat_bilstm, memory_mb=mem_bilstm)
print('\n✅ BiLSTM-15s benchmark done.')


BENCHMARK: Single-Scale BiLSTM-15s (RSU tier)
BiLSTM-15s trainable params: 312,131
  (BiLSTM branch embedding model: 312,032)
  [bilstm_15s] TF warm-up 100... done.
  [bilstm_15s] TF timed 1000... done.

Latency:
  mean_ms        : 43.3511 ms
  std_ms         : 8.4402 ms
  p50_ms         : 39.3463 ms
  p95_ms         : 61.4472 ms
  p99_ms         : 68.9960 ms
  min_ms         : 35.6492 ms
  max_ms         : 89.2054 ms

Memory:
  GPU peak: 86.0464 MB
  CPU peak: 0.0703 MB

✅ BiLSTM-15s benchmark done.


## Cell 8 — Benchmark: Full Detector (Edge server tier)

In [36]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

print('='*70)
print('BENCHMARK: Full Detector (Edge server tier)')
print('='*70)

_n_ref = min(10, processed['15s']['X_test'].shape[0])
_ref_means = processed['15s']['X_test'][:_n_ref].mean(axis=1)

def _spatial_single(x_win, k=5):
    x_mean = x_win[0].mean(axis=0).reshape(1,-1)
    sim = cos_sim(x_mean, _ref_means)[0]
    idx = np.argsort(sim)[-min(k, len(sim)):]
    ns = sim[idx]; nm = _ref_means[idx].mean(0); nstd = _ref_means[idx].std(0)
    return np.concatenate([nm, nstd, [ns.mean()], [ns.std()]]).reshape(1,-1)

def full_detector_fwd(x_inputs):
    e1  = embedding_models['1s'](x_inputs['x_1s'],   training=False).numpy()
    e5  = embedding_models['5s'](x_inputs['x_5s'],   training=False).numpy()
    e15 = embedding_models['15s'](x_inputs['x_15s'], training=False).numpy()
    sp  = _spatial_single(x_inputs['x_15s'])
    tab = _extract_one_tab(x_inputs['x_15s'])
    pxgb = xgb.predict_proba(tab)
    Z = np.hstack([e1, e5, e15, sp, pxgb])
    Z_s = sc_meta.transform(Z)
    return meta.predict_proba(Z_s)

sample_full = dict(x_1s=sample_X_1s, x_5s=sample_X_5s, x_15s=sample_X_15s)

_test = full_detector_fwd(sample_full)
print(f'Forward pass OK: output={_test.shape}, probs={np.round(_test[0], 4)}')

n_params_full_nn = sum(count_keras_params(embedding_models[f'{ws}s']) for ws in [1,5,15])
print(f'Full Detector NN params (3 BiLSTM emb): {n_params_full_nn:,}')

lat_full = measure_latency_tf(full_detector_fwd, sample_full, n_warmup=100, n_runs=1000, name='full_detector')
print('\nLatency:')
for k, v in lat_full.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

mem_full = measure_peak_memory_tf(full_detector_fwd, sample_full, name='full_detector')
print(f'\nMemory:')
print(f'  GPU peak: {mem_full["gpu_peak_mb"]:.4f} MB')
print(f'  CPU peak: {mem_full["cpu_peak_mb"]:.4f} MB')

meta_size = get_xgb_size_mb(meta)
print(f'  Meta XGBoost file size: {meta_size:.4f} MB')

RESULT_FULL = dict(config_name='Full Detector', target_tier='Edge server',
                   trainable_params_nn=n_params_full_nn,
                   meta_xgb_n_estimators=300, meta_xgb_max_depth=4,
                   latency_ms=lat_full, memory_mb=mem_full,
                   meta_model_file_size_mb=meta_size)
print('\n✅ Full Detector benchmark done.')


BENCHMARK: Full Detector (Edge server tier)
Forward pass OK: output=(1, 3), probs=[0.3333 0.3333 0.3333]
Full Detector NN params (3 BiLSTM emb): 936,096
  [full_detector] TF warm-up 100... done.
  [full_detector] TF timed 1000... done.

Latency:
  mean_ms        : 130.1520 ms
  std_ms         : 28.9095 ms
  p50_ms         : 117.4532 ms
  p95_ms         : 199.9714 ms
  p99_ms         : 215.1320 ms
  min_ms         : 107.0832 ms
  max_ms         : 223.9012 ms

Memory:
  GPU peak: 86.0464 MB
  CPU peak: 0.1095 MB
  Meta XGBoost file size: 0.3505 MB

✅ Full Detector benchmark done.


## Cell 9 — Benchmark: BERT WordPiece


**Backend:** PyTorch (bukan TF) — pakai `torch.cuda.synchronize()` untuk sync.
**Model:** DistilBertForSequenceClassification (66M params, distilbert-base-uncased)
**Input:** Single 15-second window → 5 fitur → numerik ke string token → tokenizer → MAX_LEN=128

In [37]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

print('='*70)
print('BENCHMARK: BERT WordPiece (PyTorch — identik BERT_PyTorch_v3)')
print('='*70)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BERT_MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
N_BINS  = 20

print('Loading DistilBERT tokenizer & model...')
tokenizer_bert = DistilBertTokenizerFast.from_pretrained(BERT_MODEL_NAME)
bert_model     = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=3)
bert_model = bert_model.to(DEVICE)
bert_model.eval()

n_params_bert = count_torch_params(bert_model)
n_params_bert_trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f'BERT total params    : {n_params_bert:,}')
print(f'BERT trainable params: {n_params_bert_trainable:,}')

FEATURES_BERT = ['SNR', 'RSSI', 'PDR', 'Speed', 'Relative_Speed']

def window_to_text_bert(x_padded_single, n_bins=N_BINS):
    """Convert 1 window (1, T, 5) ke string token — identik BERT_PyTorch_v3."""
    x = x_padded_single[0]
    m = ~(x == 0).all(axis=1); xv = x[m] if m.sum() > 0 else x
    tokens = []
    for c in range(xv.shape[1]):
        v = xv[:, c]
        vmin, vmax = v.min(), v.max()
        if vmax - vmin > 1e-9:
            bins = np.linspace(vmin, vmax, n_bins + 1)
            idx = np.digitize(v, bins[:-1]) - 1
            idx = np.clip(idx, 0, n_bins - 1)
        else:
            idx = np.zeros(len(v), dtype=int)
        feat_name = FEATURES_BERT[c].lower().replace('_', '')
        tokens.extend([f'{feat_name}{i}' for i in idx])
    return ' '.join(tokens)

sample_text = window_to_text_bert(sample_X_15s)
print(f'Sample text (first 120 chars): {sample_text[:120]}...')
print(f'Token count (approx): {len(sample_text.split())}')

encoding = tokenizer_bert(
    sample_text,
    truncation=True,
    padding='max_length',
    max_length=MAX_LEN,
    return_tensors='pt'
)
sample_bert_input = {
    'input_ids':      encoding['input_ids'].to(DEVICE),
    'attention_mask': encoding['attention_mask'].to(DEVICE),
}
print(f'input_ids shape: {sample_bert_input["input_ids"].shape}')

def bert_fwd_inference_only(inputs):
    """Pure model inference (tokenization already done)."""
    return bert_model(input_ids=inputs['input_ids'],
                      attention_mask=inputs['attention_mask'])

def bert_fwd_e2e(x_raw_window):
    """End-to-end: window → tokenize → BERT inference."""
    text = window_to_text_bert(x_raw_window)
    enc = tokenizer_bert(text, truncation=True, padding='max_length',
                          max_length=MAX_LEN, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in enc.items() if k in ['input_ids','attention_mask']}
    return bert_model(**inp)

with torch.no_grad():
    out = bert_fwd_inference_only(sample_bert_input)
print(f'BERT forward OK: logits shape={out.logits.shape}, '
      f'probs={torch.softmax(out.logits, dim=-1).cpu().numpy().round(4)}')

print('\n[a] Pure BERT inference (tokenization pre-done):')
lat_bert_pure = measure_latency_torch(bert_fwd_inference_only, sample_bert_input,
                                       n_warmup=100, n_runs=1000, name='bert_pure')
for k, v in lat_bert_pure.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

print('\n[b] End-to-end (window → tokenize → BERT):')
lat_bert_e2e = measure_latency_torch(bert_fwd_e2e, sample_X_15s,
                                      n_warmup=100, n_runs=1000, name='bert_e2e')
for k, v in lat_bert_e2e.items():
    if k.endswith('_ms'): print(f'  {k:15s}: {v:.4f} ms')

mem_bert = measure_peak_memory_torch(bert_fwd_inference_only, sample_bert_input, name='bert')
print(f'\nMemory:')
print(f'  GPU peak: {mem_bert["gpu_peak_mb"]:.4f} MB')
print(f'  CPU peak: {mem_bert["cpu_peak_mb"]:.4f} MB')

RESULT_BERT = dict(config_name='BERT WordPiece', target_tier='Comparison',
                   total_params=n_params_bert, trainable_params=n_params_bert_trainable,
                   model_name=BERT_MODEL_NAME, max_len=MAX_LEN, n_bins=N_BINS,
                   latency_pure_ms=lat_bert_pure, latency_e2e_ms=lat_bert_e2e,
                   memory_mb=mem_bert)
print('\n✅ BERT benchmark done.')


BENCHMARK: BERT WordPiece (PyTorch — identik BERT_PyTorch_v3)
Loading DistilBERT tokenizer & model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT total params    : 66,955,779
BERT trainable params: 66,955,779
Sample text (first 120 chars): snr19 snr19 snr19 snr18 snr18 snr17 snr17 snr16 snr16 snr15 snr15 snr15 snr15 snr14 snr14 snr14 snr14 snr14 snr14 snr14 ...
Token count (approx): 410
input_ids shape: torch.Size([1, 128])
BERT forward OK: logits shape=torch.Size([1, 3]), probs=[[0.3214 0.3498 0.3289]]

[a] Pure BERT inference (tokenization pre-done):
  [bert_pure] Torch warm-up 100... done.
  [bert_pure] Torch timed 1000... done.
  mean_ms        : 5.2996 ms
  std_ms         : 0.4975 ms
  p50_ms         : 5.1733 ms
  p95_ms         : 6.4925 ms
  p99_ms         : 7.4905 ms
  min_ms         : 4.6807 ms
  max_ms         : 9.1307 ms

[b] End-to-end (window → tokenize → BERT):
  [bert_e2e] Torch warm-up 100... done.
  [bert_e2e] Torch timed 1000... done.
  mean_ms        : 7.8697 ms
  std_ms         : 1.1163 ms
  p50_ms         : 7.5873 ms
  p95_ms         : 9.9583 ms
  p99_ms         : 13.1407 ms
  min_ms         : 6.9201 ms


## Cell 10 — Save Results + Summary

In [38]:
from datetime import datetime

results = {
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'hardware': dict(gpu_name=str(gpu_name), tf_version=tf.__version__,
                     torch_version=torch.__version__, platform='Google Colab (T4)'),
    'protocol': dict(n_warmup=100, n_timed_runs=1000, batch_size=1,
                     window_seconds=15, sync_method_tf='tf.constant(0.0).numpy()',
                     sync_method_torch='torch.cuda.synchronize()',
                     memory_gpu_tf='tf.config.experimental.get_memory_info peak',
                     memory_gpu_torch='torch.cuda.max_memory_allocated',
                     memory_cpu='tracemalloc peak',
                     benchmark_seed=BENCHMARK_SEED,
                     mode='INLINE' if _INLINE_MODE else 'STANDALONE'),
    'configurations': dict(
        full_detector   = RESULT_FULL,
        bilstm_15s   = RESULT_BILSTM,
        xgboost_only = RESULT_XGB,
        bert_wp      = RESULT_BERT,
    ),
}

with open(BENCHMARK_OUT, 'w') as f:
    json.dump(results, f, indent=2)
print(f'✅ Results saved: {BENCHMARK_OUT}')
print('\n' + '='*80)
print('  TABLE VIII — Computational Complexity (Paper-Ready)')
print('='*80)
print(f'{"Configuration":<28} {"Params":>12} {"Lat mean±std (ms)":>22} {"p95 (ms)":>9} {"GPU MB":>8} {"CPU MB":>8}')
print('-'*92)

def row(name, params_str, lat_d, mem_d):
    return (f'{name:<28} {params_str:>12} '
            f'{lat_d["mean_ms"]:>10.3f} ± {lat_d["std_ms"]:>7.3f} '
            f'{lat_d["p95_ms"]:>9.3f} '
            f'{mem_d["gpu_peak_mb"]:>8.3f} '
            f'{mem_d["cpu_peak_mb"]:>8.3f}')

print(row('Full Detector',         f'{RESULT_FULL["trainable_params_nn"]:,}',
          RESULT_FULL['latency_ms'],   RESULT_FULL['memory_mb']))
print(row('BiLSTM-15s (single)', f'{RESULT_BILSTM["trainable_params_nn"]:,}',
          RESULT_BILSTM['latency_ms'], RESULT_BILSTM['memory_mb']))
print(row('XGBoost-only (pure)', '0',
          RESULT_XGB['latency_pure_ms'], RESULT_XGB['memory_mb']))
print(row('XGBoost-only (e2e)', '0',
          RESULT_XGB['latency_e2e_ms'],  dict(gpu_peak_mb=0.0, cpu_peak_mb=0.0)))
print(row('BERT WP (pure)',      f'{RESULT_BERT["total_params"]:,}',
          RESULT_BERT['latency_pure_ms'], RESULT_BERT['memory_mb']))
print(row('BERT WP (e2e)',       f'{RESULT_BERT["total_params"]:,}',
          RESULT_BERT['latency_e2e_ms'],  RESULT_BERT['memory_mb']))
print('-'*92)

print('\n  Speedup ratios vs Full Detector:')
lat_f  = RESULT_FULL['latency_ms']['mean_ms']
lat_b  = RESULT_BILSTM['latency_ms']['mean_ms']
lat_x  = RESULT_XGB['latency_pure_ms']['mean_ms']
lat_bw = RESULT_BERT['latency_pure_ms']['mean_ms']
if lat_x  > 0: print(f'  Full Detector / XGBoost-only : {lat_f/lat_x:.1f}×')
if lat_b  > 0: print(f'  Full Detector / BiLSTM-15s   : {lat_f/lat_b:.1f}×')
if lat_bw > 0: print(f'  BERT WP    / Full Detector   : {lat_bw/lat_f:.1f}×')

gpu_f = RESULT_FULL['memory_mb']['gpu_peak_mb']
gpu_b = RESULT_BILSTM['memory_mb']['gpu_peak_mb']
gpu_bw= RESULT_BERT['memory_mb']['gpu_peak_mb']
if gpu_b  > 0: print(f'  Full Detector / BiLSTM-15s GPU mem: {gpu_f/gpu_b:.1f}×')
if gpu_bw > 0: print(f'  BERT WP    / Full Detector GPU mem: {gpu_bw/gpu_f:.1f}×')

print(f'\n  Parameter count summary:')
print(f'  Full Detector (NN only)   : {RESULT_FULL["trainable_params_nn"]:,}')
print(f'  BiLSTM-15s             : {RESULT_BILSTM["trainable_params_nn"]:,}')
print(f'  XGBoost-only           : 0 (NN params), see model file size')
print(f'  BERT WP total          : {RESULT_BERT["total_params"]:,}')
print(f'  BERT WP / Full Detector NN: {RESULT_BERT["total_params"]/max(RESULT_FULL["trainable_params_nn"],1):.1f}×')

print(f'\n✅ JSON saved to: {BENCHMARK_OUT}')


✅ Results saved: /content/drive/MyDrive/Detector_Results/complexity_benchmark_v3_25m/s.json

  TABLE VIII — Computational Complexity (Paper-Ready)
Configuration                      Params      Lat mean±std (ms)  p95 (ms)   GPU MB   CPU MB
--------------------------------------------------------------------------------------------
Full Detector                     936,096    130.152 ±  28.910   199.971   86.046    0.110
BiLSTM-15s (single)               312,131     43.351 ±   8.440    61.447   86.046    0.070
XGBoost-only (pure)                     0      0.606 ±   0.336     0.670   76.403    0.010
XGBoost-only (e2e)                      0      2.653 ±   0.595     3.004    0.000    0.000
BERT WP (pure)                 66,955,779      5.300 ±   0.498     6.493  270.753    0.043
BERT WP (e2e)                  66,955,779      7.870 ±   1.116     9.958  270.753    0.043
--------------------------------------------------------------------------------------------

  Speedup ratios vs Full De

## Cell 11 — Sanity Checks

In [39]:
print('='*70)
print('SANITY CHECKS')
print('='*70)

issues = []
lat_f  = RESULT_FULL['latency_ms']['mean_ms']
lat_b  = RESULT_BILSTM['latency_ms']['mean_ms']
lat_x  = RESULT_XGB['latency_pure_ms']['mean_ms']
lat_bw = RESULT_BERT['latency_pure_ms']['mean_ms']

if lat_x >= lat_f: issues.append('⚠ XGBoost latency >= Full Detector — periksa setup')
else: print(f'✓ XGBoost ({lat_x:.3f}ms) < Full Detector ({lat_f:.3f}ms)')

if lat_b >= lat_f: issues.append('⚠ BiLSTM-15s latency >= Full Detector — tidak wajar')
else: print(f'✓ BiLSTM-15s ({lat_b:.3f}ms) < Full Detector ({lat_f:.3f}ms)')

if lat_bw <= lat_f: issues.append(f'⚠ BERT ({lat_bw:.3f}ms) <= Full Detector ({lat_f:.3f}ms) — cek apakah BERT berjalan di CPU')
else: print(f'✓ BERT ({lat_bw:.3f}ms) > Full Detector ({lat_f:.3f}ms) — wajar (66M params)')

gf = RESULT_FULL['memory_mb']['gpu_peak_mb']
gb = RESULT_BILSTM['memory_mb']['gpu_peak_mb']
gx = RESULT_XGB['memory_mb']['gpu_peak_mb']
gbw = RESULT_BERT['memory_mb']['gpu_peak_mb']
if gx > 10: issues.append(f'⚠ XGBoost GPU mem={gx:.1f}MB — seharusnya ~0 (CPU-bound)')
else: print(f'✓ XGBoost GPU mem ≈ {gx:.2f}MB (expected ~0)')
if gf < gb: issues.append('⚠ Full Detector GPU < BiLSTM-15s GPU — tidak konsisten')
else: print(f'✓ Full Detector GPU ({gf:.1f}MB) >= BiLSTM-15s GPU ({gb:.1f}MB)')

for cfg, lat_d in [('Full Detector', RESULT_FULL['latency_ms']),
                    ('BiLSTM-15s', RESULT_BILSTM['latency_ms']),
                    ('XGBoost',    RESULT_XGB['latency_pure_ms']),
                    ('BERT WP',    RESULT_BERT['latency_pure_ms'])]:
    r = lat_d['p99_ms'] / lat_d['mean_ms'] if lat_d['mean_ms'] > 0 else 0
    if r > 5: issues.append(f'⚠ {cfg}: p99/mean={r:.1f}× — distribusi sangat skewed')
    else: print(f'✓ {cfg}: p99/mean = {r:.2f}× (acceptable)')

expected_bert_params = 66_362_883
diff = abs(RESULT_BERT['total_params'] - expected_bert_params)
if diff > 1_000_000: issues.append(f'⚠ BERT params {RESULT_BERT["total_params"]:,} jauh dari expected ~{expected_bert_params:,}')
else: print(f'✓ BERT params {RESULT_BERT["total_params"]:,} ≈ expected {expected_bert_params:,}')

print('\n' + '='*70)
if not issues:
    print('✅ Semua sanity check PASS — angka aman untuk paper.')
else:
    print(f'❌ {len(issues)} issue:')
    for i, m in enumerate(issues, 1): print(f'  {i}. {m}')
    print('\nSelesaikan issues sebelum submit.')

SANITY CHECKS
✓ XGBoost (0.606ms) < Full Detector (130.152ms)
✓ BiLSTM-15s (43.351ms) < Full Detector (130.152ms)
✓ Full Detector GPU (86.0MB) >= BiLSTM-15s GPU (86.0MB)
✓ Full Detector: p99/mean = 1.65× (acceptable)
✓ BiLSTM-15s: p99/mean = 1.59× (acceptable)
✓ XGBoost: p99/mean = 3.62× (acceptable)
✓ BERT WP: p99/mean = 1.41× (acceptable)
✓ BERT params 66,955,779 ≈ expected 66,362,883

❌ 2 issue:
  1. ⚠ BERT (5.300ms) <= Full Detector (130.152ms) — cek apakah BERT berjalan di CPU
  2. ⚠ XGBoost GPU mem=76.4MB — seharusnya ~0 (CPU-bound)

Selesaikan issues sebelum submit.
